# Krea 2 LoRA Studio · Google Colab T4

Train a Krea 2 LoRA through a lightweight browser Studio. The notebook is self-contained: setup, cache preparation, training, checkpoint previews, resume, Hugging Face upload, and direct LoRA downloads are all included.

## Before you start
1. Open this notebook in Google Colab and select **Runtime → Change runtime type → T4 GPU**.
2. Confirm your Hugging Face account can access [krea/Krea-2-Raw](https://huggingface.co/krea/Krea-2-Raw). If the model requires authentication, add a Colab Secret named `HF_TOKEN` and enable notebook access. Never place a token in a code cell or a public repository.
3. Run the notebook cells from top to bottom. Wait for setup, then open the temporary **Krea 2 · LoRA Studio** link printed by the final cell.

## Train a LoRA
1. Choose a project name and trigger word. If you want data, cache, and resume checkpoints to survive a runtime reset, enable Google Drive storage before adding the dataset.
2. Upload one ZIP with JPG, PNG, or WEBP images. Matching `.txt` captions are optional; use the same filename stem as each image. You can also select a folder already in the Colab runtime or Drive.
3. Choose **Prepare cache** and wait until it reports completion. This creates reusable image latents and caption embeddings.
4. Choose **Train**. The optional **Quick run** is a 20-step smoke check; normal runs use the configured step count. Choose **Resume** to continue from the newest complete training checkpoint.
5. In **Saved LoRA files**, refresh the list if needed. Each final/checkpoint adapter shows its step and file size with its own **Download** button. **Download all** creates one ZIP. Downloads stream directly from the Studio and can start while training continues.

## Optional Turbo previews
Enable **Generate images at each saved LoRA checkpoint** before training, then configure one or more preview cards. Each card has its own prompt and seed. Static seeds keep the sampling setup repeatable; random seeds vary at each saved checkpoint. The optional compact NF4 Krea 2 Turbo model creates 8-step previews, defaulting to 512 px. Preview weights are prepared alongside dataset caching; preview generation runs in a separate process so training can reclaim GPU memory. Preview failures do not discard a completed resume checkpoint.

## Optional Hugging Face upload
Enable upload in Studio and set the Colab `HF_TOKEN` Secret. The repository is private by default. You can also leave upload disabled and download the final adapter or individual saved versions from the Studio.

## Files and reconnects
The Studio displays training progress, system readings, stage timings, ETA, preview images, and the saved-adapter list. The temporary Studio link works only while this Colab runtime is active. Files under `/content` are temporary; Drive-backed files persist according to your Drive storage. If the runtime fully resets, run the notebook again and use **Resume** if the project output is on Drive.

## Sharing notes
This notebook contains no dataset, model weights, or access token. Each user needs their own Colab runtime and any required access to the gated base model. A free Colab session may disconnect or change GPU availability; saved Drive checkpoints allow recovery but do not prevent platform timeouts. The Cloudflare Quick Tunnel URL is temporary and is recreated when the notebook starts.

For local static validation in the source project, run `python -m unittest -v test_krea2_gui.py`. A successful local check validates notebook structure, embedded Python, and Studio HTTP behavior; it does not replace a live Colab/T4 training run.


In [ ]:
# Run once in a Colab GPU runtime. Do not replace Colab's PyTorch.
import importlib.metadata as metadata, json, subprocess, sys, time
from packaging.requirements import Requirement
_setup_started = time.monotonic()
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → T4 GPU, then run all cells again.")
print("GPU:", torch.cuda.get_device_name(0))
DIFFUSERS_COMMIT = "e0118ade2f60234c41bacf40330a7e2f61108849"
packages = [
    f"diffusers @ git+https://github.com/huggingface/diffusers.git@{DIFFUSERS_COMMIT}",
    "transformers>=4.57.0", "accelerate>=1.2.0", "peft>=0.17.0",
    "bitsandbytes>=0.46.0", "huggingface_hub>=0.34.0",
    "safetensors>=0.5.0", "sentencepiece", "ftfy", "psutil",
]
def installed_diffusers_commit():
    try:
        direct = metadata.distribution("diffusers").read_text("direct_url.json")
        return json.loads(direct or "{}").get("vcs_info", {}).get("commit_id")
    except metadata.PackageNotFoundError:
        return None

needs_install = []
current_diffusers_commit = installed_diffusers_commit()
for spec in packages:
    requirement = Requirement(spec)
    try:
        current = metadata.version(requirement.name)
        version_ok = not requirement.specifier or current in requirement.specifier
    except metadata.PackageNotFoundError:
        version_ok = False
    if requirement.name == "diffusers":
        version_ok = version_ok and current_diffusers_commit == DIFFUSERS_COMMIT
    if not version_ok:
        needs_install.append(spec)
if needs_install:
    print("Installing or repairing:", ", ".join(Requirement(spec).name for spec in needs_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *needs_install])
else:
    print("Setup packages already match the notebook pins; skipping pip install.")
# Colab's preinstalled torchao can conflict with this diffusers commit.
# The trainer uses bitsandbytes NF4, not torchao.
try:
    metadata.version("torchao")
except metadata.PackageNotFoundError:
    pass
else:
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
probe = subprocess.run([sys.executable, "-c",
    "from diffusers import AutoencoderKLQwenImage, Krea2Transformer2DModel; print('IMPORTS_OK')"],
    capture_output=True, text=True)
print(probe.stdout, probe.stderr[-2000:])
probe.check_returncode()
KREA_SETUP_SECONDS = time.monotonic() - _setup_started
print(f"Setup completed in {KREA_SETUP_SECONDS/60:.1f} min")


In [ ]:
# Verified stages. Large training and Turbo models get the T4 one at a time.
from pathlib import Path
import py_compile
PRECACHE_SOURCE = '\nimport argparse, gc, hashlib, json, math, os, shutil, time\nfrom huggingface_hub import snapshot_download\nfrom pathlib import Path\nfrom PIL import Image, ImageOps\nimport torch\nimport torchvision.transforms.functional as TVF\nfrom transformers import AutoTokenizer, Qwen3VLModel, BitsAndBytesConfig\nfrom diffusers import AutoencoderKLQwenImage\n\nIMG_EXTS={\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'}\nSELECT_LAYERS=(2,5,8,11,14,17,20,23,26,29,32,35)\nPREFIX=("<|im_start|>system\\nDescribe the image by detailing the color, shape, size, texture, quantity, text, "\n        "spatial relationships of the objects and background:<|im_end|>\\n<|im_start|>user\\n")\nSUFFIX="<|im_end|>\\n<|im_start|>assistant\\n"\nPREFIX_IDX=34\nSUFFIX_TOKENS=5\n\ndef cleanup():\n    gc.collect()\n    if torch.cuda.is_available(): torch.cuda.empty_cache()\n\ndef emit_timing(name, started):\n    print(f\'TIMING {name}={time.monotonic()-started:.2f}s\', flush=True)\n\ndef caption_key(model, max_seq, caption):\n    payload=f\'caption-cache-v1\\0{model}\\0{max_seq}\\0{caption}\'\n    return hashlib.sha256(payload.encode(\'utf-8\')).hexdigest()\n\ndef file_sha256(path):\n    digest=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef bucket_size(w,h,res,max_side=None,multiple=16):\n    target=res*res\n    ar=w/h\n    bh=math.sqrt(target/ar); bw=ar*bh\n    bw=max(multiple, round(bw/multiple)*multiple)\n    bh=max(multiple, round(bh/multiple)*multiple)\n    max_side=max_side or int(res*1.75)\n    if max(bw,bh)>max_side:\n        s=max_side/max(bw,bh)\n        bw=max(multiple, int(bw*s)//multiple*multiple)\n        bh=max(multiple, int(bh*s)//multiple*multiple)\n    return int(bw),int(bh)\n\ndef resize_crop(im,bw,bh):\n    ratio=max(bw/im.width,bh/im.height)\n    nw,nh=math.ceil(im.width*ratio),math.ceil(im.height*ratio)\n    im=im.resize((nw,nh),Image.Resampling.LANCZOS)\n    left=(nw-bw)//2; top=(nh-bh)//2\n    return im.crop((left,top,left+bw,top+bh))\n\ndef encode_text(text_encoder, tokenizer, prompts, max_len, device):\n    prompts=[prompts] if isinstance(prompts,str) else prompts\n    text=[PREFIX+p for p in prompts]\n    toks=tokenizer(text,truncation=True,padding=\'max_length\',\n                   max_length=max_len+PREFIX_IDX-SUFFIX_TOKENS,return_tensors=\'pt\').to(device)\n    suffix=tokenizer([SUFFIX]*len(text),return_tensors=\'pt\').to(device)\n    input_ids=torch.cat([toks.input_ids,suffix.input_ids],dim=1)\n    mask=torch.cat([toks.attention_mask,suffix.attention_mask],dim=1).bool()\n    pos=(mask.long().cumsum(dim=-1)-1).clamp(min=0).unsqueeze(0).expand(3,-1,-1)\n    with torch.inference_mode(), torch.autocast(\'cuda\',dtype=torch.float16):\n        out=text_encoder(input_ids=input_ids,attention_mask=mask,position_ids=pos,output_hidden_states=True)\n        hs=torch.stack([out.hidden_states[i] for i in SELECT_LAYERS],dim=2)\n    return hs[:,PREFIX_IDX:].to(torch.float16), mask[:,PREFIX_IDX:]\n\ndef main(a):\n    dataset=Path(a.dataset); cache=Path(a.cache); cache.mkdir(parents=True,exist_ok=True)\n    images=sorted([p for p in dataset.iterdir() if p.suffix.lower() in IMG_EXTS])\n    if not images: raise RuntimeError(\'No images\')\n    device=torch.device(\'cuda\')\n    meta=[]\n    latent_cache=cache/\'latent_cache\'; latent_cache.mkdir(parents=True,exist_ok=True)\n    missing_latents=[]\n    for i,p in enumerate(images,1):\n        with Image.open(p) as original:\n            im=ImageOps.exif_transpose(original).convert(\'RGB\')\n            bw,bh=bucket_size(im.width,im.height,a.resolution)\n        capfile=p.with_suffix(\'.txt\')\n        cap=capfile.read_text(encoding=\'utf-8\').strip() if a.use_txt and capfile.exists() else a.default_caption.strip()\n        if a.trigger and a.trigger.lower() not in cap.lower(): cap=f\'{a.trigger}, {cap}\'.strip(\', \')\n        stem=f\'{i:04d}\'\n        source_hash=file_sha256(p)\n        key=hashlib.sha256(f\'latent-cache-v2\\\\0{a.model}\\\\0{a.resolution}\\\\0{bw}x{bh}\\\\0{source_hash}\'.encode(\'utf-8\')).hexdigest()\n        cached_latent=latent_cache/f\'{key}.pt\'\n        item={\'id\':stem,\'source\':p.name,\'caption\':cap,\'width\':bw,\'height\':bh,\n              \'latent\':f\'{stem}_latent.pt\',\'embed\':f\'{stem}_embed.pt\',\'mask\':f\'{stem}_mask.pt\',\n              \'latent_key\':key}\n        valid=cached_latent.is_file() and cached_latent.stat().st_size>0\n        if valid:\n            try:\n                latent_check=torch.load(cached_latent,map_location=\'cpu\',weights_only=True)\n                valid=torch.is_tensor(latent_check) and latent_check.ndim==4 and latent_check.shape[0]==1\n                del latent_check\n            except Exception:\n                valid=False\n        if valid:\n            shutil.copy2(cached_latent,cache/item[\'latent\'])\n        else:\n            missing_latents.append((p,item,cached_latent))\n        meta.append(item)\n    print(f\'LATENT_CACHE {len(images)-len(missing_latents)}/{len(images)} reused; {len(missing_latents)} require encoding\')\n    if missing_latents:\n        print(\'[1/2] VAE pre-cache...\')\n        download_started=time.monotonic()\n        vae_snapshot=snapshot_download(repo_id=a.model,allow_patterns=[\'vae/*\'],token=os.environ.get(\'HF_TOKEN\'))\n        emit_timing(\'vae_download\',download_started)\n        load_started=time.monotonic()\n        vae=AutoencoderKLQwenImage.from_pretrained(vae_snapshot,subfolder=\'vae\',torch_dtype=torch.float16,low_cpu_mem_usage=True).to(device)\n        emit_timing(\'vae_load\',load_started)\n        zdim=vae.config.z_dim\n        mean=torch.tensor(vae.config.latents_mean,device=device,dtype=torch.float32).view(1,zdim,1,1,1)\n        std=torch.tensor(vae.config.latents_std,device=device,dtype=torch.float32).view(1,zdim,1,1,1)\n        encode_started=time.monotonic()\n        for j,(p,item,cached_latent) in enumerate(missing_latents,1):\n            with Image.open(p) as original:\n                im=ImageOps.exif_transpose(original).convert(\'RGB\')\n            im=resize_crop(im,item[\'width\'],item[\'height\'])\n            x=TVF.pil_to_tensor(im).unsqueeze(0).unsqueeze(2).float().div(127.5).sub(1.0).to(device,dtype=torch.float16)\n            with torch.inference_mode(), torch.autocast(\'cuda\',dtype=torch.float16):\n                z=vae.encode(x).latent_dist.sample().float()\n                z=(z-mean)/std\n            latent=z[:,:,0].to(torch.float16).cpu().contiguous()\n            temp_latent=cached_latent.with_suffix(\'.pt.tmp\')\n            torch.save(latent,temp_latent)\n            os.replace(temp_latent,cached_latent)\n            shutil.copy2(cached_latent,cache/item[\'latent\'])\n            print(f\'  VAE encode {j}/{len(missing_latents)} {p.name} -> {item["width"]}x{item["height"]}\')\n            del x,z,latent,im\n        emit_timing(\'vae_encode\',encode_started)\n        vae.to(\'cpu\'); del vae, mean, std; cleanup()\n    else:\n        print(\'TIMING vae_download=0.00s\',flush=True)\n        print(\'TIMING vae_load=0.00s\',flush=True)\n        print(\'TIMING vae_encode=0.00s\',flush=True)\n\n    print(\'[2/2] Checking reusable caption cache...\')\n    by_caption={}\n    for item in meta: by_caption.setdefault(item[\'caption\'],[]).append(item)\n    if a.preview_config:\n        preview_payload=json.loads(Path(a.preview_config).read_text(encoding=\'utf-8\'))\n        for slot in preview_payload.get(\'slots\',[]):\n            prompt=str(slot.get(\'prompt\',\'\')).strip()\n            if slot.get(\'enabled\') and prompt: by_caption.setdefault(prompt,[])\n    caption_cache=cache/\'caption_embeddings\'; caption_cache.mkdir(parents=True,exist_ok=True)\n    entries=[]; missing=[]\n    for caption,items in by_caption.items():\n        key=caption_key(a.model,a.max_seq,caption)\n        emb_file=caption_cache/f\'{key}_embed.pt\'; mask_file=caption_cache/f\'{key}_mask.pt\'\n        valid=emb_file.is_file() and mask_file.is_file() and emb_file.stat().st_size>0 and mask_file.stat().st_size>0\n        if valid:\n            try:\n                emb_check=torch.load(emb_file,map_location=\'cpu\',weights_only=True)\n                mask_check=torch.load(mask_file,map_location=\'cpu\',weights_only=True)\n                valid=emb_check.ndim==4 and mask_check.ndim==2 and emb_check.shape[0]==mask_check.shape[0]==1 and emb_check.shape[1]==mask_check.shape[1]\n                del emb_check,mask_check\n            except Exception:\n                valid=False\n        entry=(caption,items,emb_file,mask_file)\n        entries.append(entry)\n        if not valid: missing.append(entry)\n\n    tokenizer=None; text_encoder=None\n    if missing:\n        print(f\'Caption cache misses: {len(missing)}/{len(entries)}\')\n        download_started=time.monotonic()\n        qwen_snapshot=snapshot_download(repo_id=a.model,allow_patterns=[\'tokenizer/*\',\'text_encoder/*\'],token=os.environ.get(\'HF_TOKEN\'))\n        emit_timing(\'qwen_download\',download_started)\n        tokenizer_started=time.monotonic()\n        tokenizer=AutoTokenizer.from_pretrained(qwen_snapshot,subfolder=\'tokenizer\')\n        emit_timing(\'tokenizer_load\',tokenizer_started)\n        qwen_started=time.monotonic()\n        text_encoder=Qwen3VLModel.from_pretrained(qwen_snapshot,subfolder=\'text_encoder\',torch_dtype=torch.float16,low_cpu_mem_usage=True,quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True),device_map={\'\':0})\n        emit_timing(\'qwen_model_load\',qwen_started)\n        text_encoder.eval()\n    caption_started=time.monotonic()\n    missing_keys={caption for caption,items,emb_file,mask_file in missing}\n    for j,(caption,items,emb_file,mask_file) in enumerate(entries,1):\n        if caption in missing_keys:\n            emb,mask=encode_text(text_encoder,tokenizer,caption,a.max_seq,device)\n            emb=emb.cpu().contiguous(); mask=mask.cpu().contiguous()\n            torch.save(emb,emb_file); torch.save(mask,mask_file)\n            del emb,mask\n        for item in items:\n            shutil.copyfile(emb_file,cache/item[\'embed\'])\n            shutil.copyfile(mask_file,cache/item[\'mask\'])\n        reused=\'reused\' if caption not in missing_keys else \'encoded\'\n        print(f\'  caption {j}/{len(entries)} {reused} -> {len(items)} image(s)\')\n    emit_timing(\'caption_cache\',caption_started)\n    if text_encoder is not None:\n        del text_encoder,tokenizer; cleanup()\n    (cache/\'metadata.json\').write_text(json.dumps({\'model\':a.model,\'resolution\':a.resolution,\'max_seq\':a.max_seq,\'items\':meta},ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(\'CACHE_READY\', cache/\'metadata.json\')\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--model\',default=\'krea/Krea-2-Raw\'); ap.add_argument(\'--dataset\',required=True); ap.add_argument(\'--cache\',required=True)\n    ap.add_argument(\'--resolution\',type=int,default=512); ap.add_argument(\'--max-seq\',type=int,default=128)\n    ap.add_argument(\'--trigger\',default=\'\'); ap.add_argument(\'--default-caption\',default=\'photo of a person\')\n    ap.add_argument(\'--use-txt\',action=\'store_true\'); ap.add_argument(\'--preview-config\',default=\'\')\n    main(ap.parse_args())\n'
TRAIN_SOURCE = "\nimport argparse, gc, json, math, os, random, shutil, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nimport bitsandbytes as bnb\nfrom diffusers import BitsAndBytesConfig, FlowMatchEulerDiscreteScheduler, Krea2Pipeline, Krea2Transformer2DModel\nfrom diffusers.training_utils import cast_training_params\nfrom diffusers.utils import convert_unet_state_dict_to_peft\nfrom peft import LoraConfig, prepare_model_for_kbit_training, set_peft_model_state_dict\nfrom peft.utils import get_peft_model_state_dict\nfrom huggingface_hub import HfApi, snapshot_download\n\n\ndef cleanup():\n    gc.collect(); torch.cuda.empty_cache()\n\ndef emit_timing(name, started):\n    print(f'TIMING {name}={time.monotonic()-started:.2f}s', flush=True)\n\ndef pack_latents(x):\n    B,C,H,W=x.shape; H-=H%2; W-=W%2; x=x[:,:,:H,:W]\n    return x.view(B,C,H//2,2,W//2,2).permute(0,2,4,1,3,5).reshape(B,(H//2)*(W//2),C*4)\n\ndef calculate_shift(seq,base_seq=256,max_seq=6400,base_shift=.5,max_shift=1.15):\n    m=(max_shift-base_shift)/(max_seq-base_seq); b=base_shift-m*base_seq\n    return seq*m+b\n\ndef sample_sigma(batch,seq,device,cfg):\n    u=torch.rand(batch,device=device).clamp(1e-6,1-1e-6)\n    mu=calculate_shift(seq,*cfg); em=math.exp(mu)\n    return (em/(em+(1.0/u-1.0))).clamp(1e-4,1-1e-4)\n\ndef save_lora(model,path):\n    path=Path(path); path.mkdir(parents=True,exist_ok=True)\n    Krea2Pipeline.save_lora_weights(str(path),transformer_lora_layers=get_peft_model_state_dict(model),safe_serialization=True)\n\ndef load_lora(model,path):\n    sd=Krea2Pipeline.lora_state_dict(str(path))\n    sd={k.replace('transformer.','',1):v for k,v in sd.items() if k.startswith('transformer.')}\n    sd=convert_unet_state_dict_to_peft(sd)\n    bad=set_peft_model_state_dict(model,sd,adapter_name='default')\n    unexpected=getattr(bad,'unexpected_keys',None) if bad is not None else None\n    if unexpected: print('Warning unexpected LoRA keys:',unexpected[:10])\n\ndef save_checkpoint(model,opt,scaler,out,step,keep=2,api=None,repo_id=None):\n    checkpoint_started=time.monotonic()\n    out=Path(out); ck=out/f'checkpoint-{step}'; tmp_ck=out/f'.checkpoint-{step}.pending'\n    if tmp_ck.exists(): shutil.rmtree(tmp_ck,ignore_errors=True)\n    tmp_ck.mkdir(parents=True,exist_ok=False)\n    save_lora(model,tmp_ck)\n    state={'step':step,'optimizer':opt.state_dict(),'scaler':scaler.state_dict(),\n           'python_rng':random.getstate(),'torch_rng':torch.get_rng_state(),\n           'cuda_rng':torch.cuda.get_rng_state_all()}\n    state_tmp=tmp_ck/'training_state.pt.tmp'\n    torch.save(state,state_tmp)\n    os.replace(state_tmp,tmp_ck/'training_state.pt')\n    adapter=tmp_ck/'pytorch_lora_weights.safetensors'\n    state_file=tmp_ck/'training_state.pt'\n    checkpoint_meta={'schema':1,'step':step,'adapter_bytes':adapter.stat().st_size,\n                     'state_bytes':state_file.stat().st_size,'saved_at':time.time()}\n    meta_tmp=tmp_ck/'checkpoint.json.tmp'\n    meta_tmp.write_text(json.dumps(checkpoint_meta,indent=2),encoding='utf-8')\n    os.replace(meta_tmp,tmp_ck/'checkpoint.json')\n\n    backup_ck=out/f'.checkpoint-{step}.previous'\n    if ck.exists():\n        if backup_ck.exists(): shutil.rmtree(backup_ck,ignore_errors=True)\n        os.replace(ck,backup_ck)\n    try:\n        os.replace(tmp_ck,ck)\n    except Exception:\n        if backup_ck.exists() and not ck.exists(): os.replace(backup_ck,ck)\n        raise\n    if backup_ck.exists(): shutil.rmtree(backup_ck,ignore_errors=True)\n\n    latest=out/'latest_checkpoint.json'\n    latest_tmp=out/'latest_checkpoint.json.tmp'\n    latest_tmp.write_text(json.dumps({'step':step,'path':ck.name,'saved_at':time.time()},indent=2),encoding='utf-8')\n    os.replace(latest_tmp,latest)\n\n    # Permanent LoRA snapshots survive cleanup of heavy resume checkpoints.\n    versions=out/'versions'; versions.mkdir(parents=True,exist_ok=True)\n    src=ck/'pytorch_lora_weights.safetensors'\n    version_file=versions/f'lora_step_{step:05d}.safetensors'\n    version_tmp=versions/f'.{version_file.name}.pending'\n    try:\n        shutil.copy2(src,version_tmp)\n        if version_tmp.stat().st_size!=src.stat().st_size: raise RuntimeError(f'Incomplete LoRA snapshot copy: {version_tmp}')\n        os.replace(version_tmp,version_file)\n    finally:\n        version_tmp.unlink(missing_ok=True)\n    print(f'  Saved permanent LoRA snapshot: {version_file.name}')\n\n    # Keep only complete numeric optimizer/RNG checkpoints.\n    cks=[]\n    for path in out.glob('checkpoint-*'):\n        try: checkpoint_step=int(path.name.split('-')[-1])\n        except ValueError: continue\n        if path.is_dir() and (path/'training_state.pt').is_file() and (path/'pytorch_lora_weights.safetensors').is_file():\n            cks.append((checkpoint_step,path))\n    cks.sort(key=lambda row:row[0])\n    keep=max(1,int(keep))\n    for _,old in cks[:-keep]: shutil.rmtree(old,ignore_errors=True)\n\n    print(f'TIMING checkpoint_local={time.monotonic()-checkpoint_started:.2f}s',flush=True)\n    # Hub checkpoints are permanent LoRA snapshots; full optimizer state remains on Drive/local storage.\n    if api is not None and repo_id:\n        upload_started=time.monotonic()\n        api.upload_file(path_or_fileobj=str(version_file),\n                        path_in_repo=f'checkpoints/{version_file.name}',\n                        repo_id=repo_id,repo_type='model',\n                        commit_message=f'LoRA checkpoint step {step}')\n        progress=out/'hub_progress.json'\n        progress.write_text(json.dumps({'latest_step':step,'latest_file':version_file.name,'updated_at':time.time()},indent=2),encoding='utf-8')\n        api.upload_file(path_or_fileobj=str(progress),path_in_repo='progress.json',repo_id=repo_id,repo_type='model')\n        print(f'  Uploaded to HF: {repo_id}/checkpoints/{version_file.name}')\n        emit_timing('checkpoint_upload',upload_started)\n    return ck,version_file\n\n\ndef newest_checkpoint(out):\n    out=Path(out)\n    # Recover a fully written checkpoint if a runtime stopped during the final directory rename.\n    for backup in out.glob('.checkpoint-*.previous'):\n        try: step=int(backup.name[len('.checkpoint-'):].split('.',1)[0])\n        except ValueError: continue\n        target=out/f'checkpoint-{step}'\n        if not target.exists() and (backup/'training_state.pt').is_file() and (backup/'pytorch_lora_weights.safetensors').is_file():\n            os.replace(backup,target)\n    cks=[]\n    for path in out.glob('checkpoint-*'):\n        try: step=int(path.name.split('-')[-1])\n        except ValueError: continue\n        state=path/'training_state.pt'; adapter=path/'pytorch_lora_weights.safetensors'\n        if not path.is_dir() or not state.is_file() or not adapter.is_file(): continue\n        meta_file=path/'checkpoint.json'\n        if meta_file.is_file():\n            try:\n                meta=json.loads(meta_file.read_text(encoding='utf-8'))\n                if meta.get('step')!=step or meta.get('state_bytes')!=state.stat().st_size or meta.get('adapter_bytes')!=adapter.stat().st_size:\n                    continue\n            except Exception: continue\n        cks.append(path)\n    return max(cks,key=lambda path:int(path.name.split('-')[-1])) if cks else None\n\ndef main(a):\n    metadata_started=time.monotonic()\n    device=torch.device('cuda')\n    random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)\n    meta=json.loads((Path(a.cache)/'metadata.json').read_text(encoding='utf-8'))\n    items=meta['items']; Path(a.output).mkdir(parents=True,exist_ok=True)\n    emit_timing('metadata_load',metadata_started)\n\n    hub_started=time.monotonic()\n    api=None; repo_id=None\n    if a.push_to_hub:\n        token=os.environ.get('HF_TOKEN')\n        if not token:\n            raise RuntimeError('PUSH_TO_HUB requested but HF_TOKEN is missing')\n        api=HfApi(token=token)\n        repo_id=a.hub_model_id.strip()\n        if not repo_id:\n            username=api.whoami()['name']\n            repo_id=f'{username}/{Path(a.output).name}'\n        api.create_repo(repo_id=repo_id,repo_type='model',private=a.private_repo,exist_ok=True)\n        print('HF checkpoint repository:',repo_id)\n    emit_timing('hub_setup',hub_started)\n\n    print('Downloading transformer weights if needed...')\n    download_started=time.monotonic()\n    transformer_snapshot=snapshot_download(repo_id=a.model,allow_patterns=['transformer/*'],token=os.environ.get('HF_TOKEN'))\n    emit_timing('transformer_download',download_started)\n    print('Loading NF4 Krea-2 transformer from local cache...')\n    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)\n    model_load_started=time.monotonic()\n    model=Krea2Transformer2DModel.from_pretrained(transformer_snapshot,subfolder='transformer',quantization_config=q,\n            torch_dtype=torch.float16,low_cpu_mem_usage=True,device_map={'':0})\n    emit_timing('transformer_nf4_load',model_load_started)\n    adapter_started=time.monotonic()\n    model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=False)\n    model.requires_grad_(False)\n    if a.grad_ckpt:\n        model.enable_gradient_checkpointing()\n        def req_grad(_m,_i,o):\n            if torch.is_tensor(o): o.requires_grad_(True)\n        model.img_in.register_forward_hook(req_grad)\n\n    if a.attention_only:\n        targets=['to_q','to_k','to_v','to_out.0','to_gate']\n    else:\n        targets=['img_in','final_layer.linear','to_q','to_k','to_v','to_out.0','to_gate','ff.up','ff.down',\n                 'text_fusion.projector','txt_in.linear_1','txt_in.linear_2','time_embed.linear_1','time_embed.linear_2','time_mod_proj']\n    model.add_adapter(LoraConfig(r=a.rank,lora_alpha=a.rank,lora_dropout=0.0,init_lora_weights='gaussian',target_modules=targets))\n    cast_training_params([model],dtype=torch.float32)\n    trainable=[p for p in model.parameters() if p.requires_grad]\n    n=sum(p.numel() for p in trainable)\n    print(f'Trainable LoRA params: {n:,} ({n/1e6:.1f}M)')\n    opt=bnb.optim.AdamW8bit(trainable,lr=a.lr,weight_decay=0.0)\n    try: scaler=torch.amp.GradScaler('cuda',enabled=True)\n    except Exception: scaler=torch.cuda.amp.GradScaler(enabled=True)\n\n    sched=FlowMatchEulerDiscreteScheduler.from_pretrained(a.model,subfolder='scheduler')\n    shift_cfg=(sched.config.get('base_image_seq_len',256),sched.config.get('max_image_seq_len',6400),\n               sched.config.get('base_shift',.5),sched.config.get('max_shift',1.15))\n    emit_timing('adapter_optimizer_setup',adapter_started)\n\n    # Small cached dataset only; keep it in pinned CPU memory for fast H2D.\n    cache_started=time.monotonic()\n    cache=[]\n    for it in items:\n        lat=torch.load(Path(a.cache)/it['latent'],map_location='cpu',weights_only=True).to(torch.float16)\n        emb=torch.load(Path(a.cache)/it['embed'],map_location='cpu',weights_only=True).to(torch.float16)\n        mask=torch.load(Path(a.cache)/it['mask'],map_location='cpu',weights_only=True).bool()\n        try: lat,emb,mask=lat.pin_memory(),emb.pin_memory(),mask.pin_memory()\n        except Exception: pass\n        cache.append((it,lat,emb,mask))\n    emit_timing('cache_ram_load',cache_started)\n\n    start=0\n    if a.resume:\n        ck=newest_checkpoint(a.output)\n        if ck:\n            print('Resuming:',ck)\n            load_lora(model,ck)\n            st=torch.load(ck/'training_state.pt',map_location='cpu',weights_only=False)\n            opt.load_state_dict(st['optimizer']); scaler.load_state_dict(st['scaler']); start=int(st['step'])\n            random.setstate(st['python_rng']); torch.set_rng_state(st['torch_rng']); torch.cuda.set_rng_state_all(st['cuda_rng'])\n\n    model.train(); opt.zero_grad(set_to_none=True)\n    torch.cuda.reset_peak_memory_stats()\n    train_loop_started=time.monotonic()\n    pos_cache={}; step_durations=[]; losses=[]\n    for step in range(start+1,a.steps+1):\n        t0=time.monotonic(); accum_loss=0.0\n        for micro in range(a.grad_accum):\n            it,lat,emb,mask=random.choice(cache)\n            lat=lat.to(device,non_blocking=True); emb=emb.to(device,non_blocking=True); mask=mask.to(device,non_blocking=True)\n            packed=pack_latents(lat); B,seq,_=packed.shape\n            sigma=sample_sigma(B,seq,device,shift_cfg); s=sigma.view(-1,1,1)\n            noise=torch.randn_like(packed); noisy=((1-s)*packed+s*noise).to(torch.float16); target=noise-packed\n            key=(emb.shape[1],lat.shape[2],lat.shape[3])\n            if key not in pos_cache:\n                pos_cache[key]=Krea2Pipeline.prepare_position_ids(emb.shape[1],lat.shape[2]//2,lat.shape[3]//2,device)\n            with torch.autocast('cuda',dtype=torch.float16):\n                pred=model(hidden_states=noisy,encoder_hidden_states=emb,timestep=sigma,position_ids=pos_cache[key],\n                           encoder_attention_mask=mask,return_dict=False)[0]\n                loss=F.mse_loss(pred.float(),target.float())/a.grad_accum\n            scaler.scale(loss).backward(); accum_loss+=float(loss.detach())*a.grad_accum\n            del lat,emb,mask,packed,noise,noisy,target,pred,loss\n        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable,1.0)\n        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)\n        dt=time.monotonic()-t0; step_durations.append(dt); losses.append(accum_loss)\n        if step==start+1: emit_timing('first_step_compute',t0)\n        steady=step_durations[1:][-10:] or step_durations[-1:]\n        ema=sum(steady)/len(steady)\n        eta=(a.steps-step)*ema\n        peak=torch.cuda.max_memory_allocated()/1024**3\n        print(f'\\rstep {step}/{a.steps} | loss {sum(losses[-20:])/len(losses[-20:]):.4f} | {ema:.2f}s/step | peak {peak:.2f}GB | ETA {eta/60:.1f}m',end='',flush=True)\n        if a.save_every>0 and step%a.save_every==0:\n            print(); save_checkpoint(model,opt,scaler,a.output,step,a.keep_checkpoints,api,repo_id)\n            if a.preview_at_checkpoints and step<a.steps:\n                print(f'PREVIEW_REQUIRED step={step}',flush=True)\n                raise SystemExit(75)\n    print()\n    emit_timing('train_loop_total',train_loop_started)\n    steady_steps=step_durations[1:]\n    if steady_steps: print(f'TIMING train_steady_sec_per_step={sum(steady_steps)/len(steady_steps):.2f}s',flush=True)\n    # Always make a permanent snapshot at the final step, even if it is not divisible by SAVE_EVERY.\n    if not (a.save_every>0 and a.steps%a.save_every==0):\n        save_checkpoint(model,opt,scaler,a.output,a.steps,a.keep_checkpoints,api,repo_id)\n    export_started=time.monotonic()\n    save_lora(model,a.output)\n    cfg=vars(a).copy(); cfg['peak_vram_gb']=torch.cuda.max_memory_allocated()/1024**3\n    cfg_path=Path(a.output)/'training_config.json'\n    cfg_path.write_text(json.dumps(cfg,indent=2),encoding='utf-8')\n    final_file=Path(a.output)/'pytorch_lora_weights.safetensors'\n    emit_timing('final_export_save',export_started)\n    if api is not None and repo_id:\n        upload_started=time.monotonic()\n        api.upload_file(path_or_fileobj=str(final_file),path_in_repo='pytorch_lora_weights.safetensors',repo_id=repo_id,repo_type='model',commit_message='Final LoRA')\n        api.upload_file(path_or_fileobj=str(cfg_path),path_in_repo='training_config.json',repo_id=repo_id,repo_type='model')\n        emit_timing('final_upload',upload_started)\n        print('Final LoRA uploaded:',f'https://huggingface.co/{repo_id}')\n    print('DONE. Final LoRA:',final_file)\n    print('Permanent versions:',Path(a.output)/'versions')\n\nif __name__=='__main__':\n    ap=argparse.ArgumentParser(); ap.add_argument('--model',default='krea/Krea-2-Raw'); ap.add_argument('--cache',required=True); ap.add_argument('--output',required=True)\n    ap.add_argument('--steps',type=int,default=20); ap.add_argument('--rank',type=int,default=8); ap.add_argument('--lr',type=float,default=3e-4)\n    ap.add_argument('--seed',type=int,default=42); ap.add_argument('--save-every',type=int,default=50); ap.add_argument('--keep-checkpoints',type=int,default=2); ap.add_argument('--preview-at-checkpoints',action='store_true')\n    ap.add_argument('--grad-accum',type=int,default=1); ap.add_argument('--attention-only',action='store_true'); ap.add_argument('--grad-ckpt',action='store_true'); ap.add_argument('--resume',action='store_true')\n    ap.add_argument('--push-to-hub',action='store_true'); ap.add_argument('--hub-model-id',default=''); ap.add_argument('--private-repo',action='store_true')\n    main(ap.parse_args())\n"
PREVIEW_SOURCE = '#!/usr/bin/env python3\n"""One-checkpoint compact NF4 Turbo preview worker for the Krea 2 Studio notebook."""\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport os\nimport secrets\nimport shutil\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom fnmatch import fnmatchcase\nfrom pathlib import Path\n\nimport torch\nfrom huggingface_hub import snapshot_download\nfrom huggingface_hub.errors import LocalEntryNotFoundError\nfrom transformers import BitsAndBytesConfig\nfrom diffusers import AutoencoderKLQwenImage, Krea2Pipeline, Krea2Transformer2DModel\n\nTURBO_MODEL = "OzzyGT/Krea_2_Turbo_bnb_nf4"\nTURBO_MODEL_REVISION = "5458debf8356a6646a5aa814de28dcea881f8a6d"\nTRAIN_MODEL = "krea/Krea-2-Raw"\nPREVIEW_MODEL_MIN_BYTES = 8 * 1024**3\nPREVIEW_DISK_RESERVE_BYTES = 2 * 1024**3\n\n\nclass IncompletePreviewSnapshot(RuntimeError):\n    pass\n\n\ndef validate_snapshot(snapshot, *, component):\n    root = Path(snapshot)\n    if component == "Turbo":\n        required = (root / "model_index.json", root / "scheduler" / "scheduler_config.json",\n                    root / "transformer" / "config.json")\n        weight_root = root / "transformer"\n    else:\n        required = (root / "vae" / "config.json",)\n        weight_root = root / "vae"\n    missing = [str(path.relative_to(root)) for path in required if not path.is_file()]\n    weights = [path for path in weight_root.rglob("*.safetensors") if path.is_file() and path.stat().st_size > 0]\n    if missing or not weights:\n        details = ", ".join(missing + ([] if weights else [f"{weight_root.relative_to(root)}/*.safetensors"]))\n        raise IncompletePreviewSnapshot(f"{component} snapshot is incomplete at {root}: missing {details}")\n    return root\n\n\ndef expected_snapshot_bytes(repo_id, revision, allow_patterns, token):\n    from huggingface_hub import HfApi\n\n    info_args = {"repo_id": repo_id, "files_metadata": True}\n    if revision:\n        info_args["revision"] = revision\n    siblings = HfApi(token=token).model_info(**info_args).siblings\n    selected = [item for item in siblings\n                if any(fnmatchcase(item.rfilename, pattern) for pattern in allow_patterns)]\n    if not selected:\n        return None\n    sizes = []\n    for item in selected:\n        size = getattr(item, "size", None)\n        if size is None:\n            size = getattr(getattr(item, "lfs", None), "size", None)\n        if size is None:\n            return None\n        sizes.append(int(size))\n    return sum(sizes)\n\n\ndef resolve_snapshot(repo_id, *, revision=None, allow_patterns, token, component, min_free_bytes):\n    request = {"repo_id": repo_id, "allow_patterns": allow_patterns, "token": token}\n    if revision:\n        request["revision"] = revision\n    try:\n        snapshot = snapshot_download(**request, local_files_only=True)\n        return validate_snapshot(snapshot, component=component)\n    except (LocalEntryNotFoundError, IncompletePreviewSnapshot) as local_error:\n        try:\n            expected = expected_snapshot_bytes(repo_id, revision, allow_patterns, token)\n        except Exception as exc:\n            raise RuntimeError(\n                f"{component} preview cache is incomplete, and Hugging Face could not confirm the download size. "\n                "Prepare the preview cache again when the connection is available."\n            ) from exc\n        if expected is None:\n            raise RuntimeError(\n                f"{component} preview cache is incomplete, and the download size could not be verified. "\n                "Prepare the preview cache again before training."\n            ) from local_error\n        required = max(int(min_free_bytes), expected) + PREVIEW_DISK_RESERVE_BYTES\n        free = shutil.disk_usage("/content").free\n        if free < required:\n            raise RuntimeError(\n                f"{component} preview files are missing from the local cache and cannot be downloaded safely: "\n                f"{free / 1024**3:.1f} GiB free, {required / 1024**3:.1f} GiB required including reserve. "\n                "Free Colab disk space, then prepare the preview cache again."\n            ) from local_error\n        print(f"{component} snapshot is missing or incomplete; downloading {expected / 1024**3:.1f} GiB of pinned files once ({free / 1024**3:.1f} GiB free)...", flush=True)\n        snapshot = snapshot_download(**request, local_files_only=False)\n        return validate_snapshot(snapshot, component=component)\n\n\ndef write_manifest(path, states):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps({"schema": 1, "slots": states}, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef mark_failed(manifest_path, exc):\n    try:\n        value = json.loads(manifest_path.read_text(encoding="utf-8"))\n        states = value.get("slots", {})\n        for state in states.values():\n            if state.get("status") in {"loading", "generating"}:\n                state.update({"status": "error", "updated_at": time.time(),\n                              "error": f"{type(exc).__name__}: {exc}"[:400]})\n        write_manifest(manifest_path, states)\n    except (OSError, ValueError, json.JSONDecodeError):\n        pass\n\n\ndef save_preview_files(image, version, latest):\n    """Atomically publish a PNG and update latest with a hard link when supported."""\n    started = time.monotonic()\n    version = Path(version)\n    latest = Path(latest)\n    version.parent.mkdir(parents=True, exist_ok=True)\n    temporary_version = version.with_name(f".{version.stem}.{os.getpid()}.tmp.png")\n    temporary_latest = latest.with_name(f".{latest.stem}.{os.getpid()}.tmp.png")\n    try:\n        image.save(temporary_version, format="PNG", compress_level=1)\n        os.replace(temporary_version, version)\n        try:\n            os.link(version, temporary_latest)\n        except OSError:\n            shutil.copy2(version, temporary_latest)\n        os.replace(temporary_latest, latest)\n    finally:\n        for temporary in (temporary_version, temporary_latest):\n            try:\n                temporary.unlink()\n            except FileNotFoundError:\n                pass\n    return time.monotonic() - started\n\n\ndef main(args):\n    started = time.monotonic()\n    preview_root = Path(args.output) / "previews"\n    manifest_path = preview_root / "latest.json"\n    config = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    slots = [slot for slot in config.get("slots", [])\n             if slot.get("enabled") and str(slot.get("prompt", "")).strip()]\n    if not slots:\n        print("No enabled Turbo preview cards.", flush=True)\n        return\n\n    states = {}\n    for slot in slots:\n        number = int(slot["slot"])\n        states[str(number)] = {"slot": number, "status": "loading", "step": int(args.step),\n                               "prompt": str(slot["prompt"]), "updated_at": time.time()}\n    write_manifest(manifest_path, states)\n\n    cache = Path(args.cache)\n    metadata = json.loads((cache / "metadata.json").read_text(encoding="utf-8"))\n    max_seq = int(metadata["max_seq"])\n    embeddings = {}\n    for slot in slots:\n        prompt = str(slot["prompt"]).strip()\n        payload = "caption-cache-v1" + chr(0) + "krea/Krea-2-Raw" + chr(0) + str(max_seq) + chr(0) + prompt\n        key = hashlib.sha256(payload.encode("utf-8")).hexdigest()\n        embedding_file = cache / "caption_embeddings" / f"{key}_embed.pt"\n        mask_file = cache / "caption_embeddings" / f"{key}_mask.pt"\n        if not embedding_file.is_file() or not mask_file.is_file():\n            raise FileNotFoundError(f"Cached prompt embeddings are missing for Preview card {slot[\'slot\']}; run Prepare cache again.")\n        embedding = torch.load(embedding_file, map_location="cpu", weights_only=True)\n        mask = torch.load(mask_file, map_location="cpu", weights_only=True)\n        if (not torch.is_tensor(embedding) or not torch.is_tensor(mask) or\n                embedding.ndim != 4 or mask.ndim != 2 or embedding.shape[0] != 1 or mask.shape[0] != 1 or\n                embedding.shape[1] != mask.shape[1]):\n            raise ValueError(f"Cached prompt embeddings are invalid for Preview card {slot[\'slot\']}.")\n        embedding = embedding.to(torch.float16).contiguous()\n        mask = mask.bool().contiguous()\n        try:\n            embedding = embedding.pin_memory()\n            mask = mask.pin_memory()\n        except RuntimeError:\n            pass\n        embeddings[int(slot["slot"])] = (embedding, mask)\n\n    print(f"Loading compact NF4 Turbo for checkpoint {args.step}...", flush=True)\n    model_started = time.monotonic()\n    download_started = time.monotonic()\n    snapshot = resolve_snapshot(\n        TURBO_MODEL, revision=TURBO_MODEL_REVISION,\n        allow_patterns=["model_index.json", "scheduler/*", "transformer/*"],\n        token=os.environ.get("HF_TOKEN"), component="Turbo", min_free_bytes=PREVIEW_MODEL_MIN_BYTES)\n    print(f"TIMING preview_download={time.monotonic() - download_started:.2f}s", flush=True)\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                      bnb_4bit_compute_dtype=torch.float16,\n                                      bnb_4bit_use_double_quant=False)\n    transformer_started = time.monotonic()\n    transformer = Krea2Transformer2DModel.from_pretrained(\n        snapshot, subfolder="transformer", quantization_config=quantization,\n        torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map={"": 0})\n    print(f"TIMING preview_nf4_load={time.monotonic() - transformer_started:.2f}s", flush=True)\n    vae_download_started = time.monotonic()\n    vae_snapshot = resolve_snapshot(\n        TRAIN_MODEL, allow_patterns=["vae/*"], token=os.environ.get("HF_TOKEN"),\n        component="Krea 2 VAE", min_free_bytes=1024**3)\n    print(f"TIMING preview_vae_download={time.monotonic() - vae_download_started:.2f}s", flush=True)\n    vae_started = time.monotonic()\n    vae = AutoencoderKLQwenImage.from_pretrained(\n        vae_snapshot, subfolder="vae", torch_dtype=torch.float16, low_cpu_mem_usage=True)\n    print(f"TIMING preview_vae_load={time.monotonic() - vae_started:.2f}s", flush=True)\n    pipeline_started = time.monotonic()\n    pipe = Krea2Pipeline.from_pretrained(\n        snapshot, tokenizer=None, text_encoder=None, transformer=transformer, vae=vae,\n        torch_dtype=torch.float16, low_cpu_mem_usage=True)\n    pipe.vae.to("cuda")\n    pipe.set_progress_bar_config(disable=True)\n    print(f"TIMING preview_pipeline_load={time.monotonic() - pipeline_started:.2f}s", flush=True)\n    adapter_started = time.monotonic()\n    pipe.load_lora_weights(str(Path(args.adapter).parent), weight_name=Path(args.adapter).name,\n                           adapter_name="studio_preview")\n    print(f"TIMING preview_adapter_load={time.monotonic() - adapter_started:.2f}s", flush=True)\n    print(f"TIMING preview_model_load={time.monotonic() - model_started:.2f}s", flush=True)\n\n    failures = []\n    image_seconds_total = 0.0\n    image_save_work_total = 0.0\n    save_executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="krea-preview-save")\n    pending_saves = {}\n\n    def finish_preview_save(number, wait):\n        nonlocal image_save_work_total\n        future, state, image_seconds, filename = pending_saves[number]\n        if not wait and not future.done():\n            return False\n        try:\n            save_seconds = future.result()\n        except Exception as exc:\n            failures.append((number, str(exc)))\n            state.update({"status": "error", "updated_at": time.time(),\n                          "error": f"Image save failed: {type(exc).__name__}: {exc}"[:400]})\n            print(f"PREVIEW_ERROR slot={number}: image save failed: {type(exc).__name__}: {exc}", flush=True)\n        else:\n            image_save_work_total += save_seconds\n            state.update({"status": "ready", "updated_at": time.time(), "filename": filename,\n                          "seconds": image_seconds, "save_seconds": save_seconds})\n            print(f"TIMING preview_save_{number}={save_seconds:.2f}s", flush=True)\n        states[str(number)] = state\n        write_manifest(manifest_path, states)\n        del pending_saves[number]\n        return True\n\n    try:\n        for index, slot in enumerate(slots, 1):\n            for saved_number in list(pending_saves):\n                finish_preview_save(saved_number, wait=False)\n            number = int(slot["slot"])\n            prompt = str(slot["prompt"]).strip()\n            seed = secrets.randbelow(2**32) if slot.get("random_seed") else int(slot.get("seed", 42))\n            state = {"slot": number, "status": "generating", "step": int(args.step),\n                     "prompt": prompt, "seed": seed, "updated_at": time.time()}\n            states[str(number)] = state\n            write_manifest(manifest_path, states)\n            print(f"preview {index}/{len(slots)} · slot {number} · step {args.step} · seed {seed}", flush=True)\n            image_started = time.monotonic()\n            generated = prompt_embeds = prompt_mask = generator = None\n            try:\n                prompt_embeds, prompt_mask = embeddings[number]\n                generator = torch.Generator(device="cuda").manual_seed(seed)\n                with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):\n                    generated = pipe(prompt_embeds=prompt_embeds.to("cuda", non_blocking=True),\n                                     prompt_embeds_mask=prompt_mask.to("cuda", non_blocking=True),\n                                     height=int(args.resolution), width=int(args.resolution),\n                                     num_inference_steps=8, guidance_scale=0.0,\n                                     generator=generator).images[0]\n            except Exception as exc:\n                image_seconds_total += time.monotonic() - image_started\n                failures.append((number, str(exc)))\n                del generated, prompt_embeds, prompt_mask, generator\n                state.update({"status": "error", "updated_at": time.time(), "error": str(exc)[:400]})\n                states[str(number)] = state\n                write_manifest(manifest_path, states)\n                print(f"PREVIEW_ERROR slot={number}: {type(exc).__name__}: {exc}", flush=True)\n                continue\n\n            image_seconds = time.monotonic() - image_started\n            image_seconds_total += image_seconds\n            folder = preview_root / f"slot_{number}"\n            folder.mkdir(parents=True, exist_ok=True)\n            version = folder / f"step_{int(args.step):05d}.png"\n            latest = folder / "latest.png"\n            future = save_executor.submit(save_preview_files, generated, version, latest)\n            pending_saves[number] = (future, state, image_seconds, version.name)\n            state.update({"status": "saving", "updated_at": time.time(),\n                          "seconds": image_seconds, "filename": version.name})\n            states[str(number)] = state\n            write_manifest(manifest_path, states)\n            del generated, prompt_embeds, prompt_mask, generator\n            print(f"TIMING preview_image_{number}={image_seconds:.2f}s", flush=True)\n        for saved_number in list(pending_saves):\n            finish_preview_save(saved_number, wait=True)\n    finally:\n        try:\n            for saved_number in list(pending_saves):\n                finish_preview_save(saved_number, wait=True)\n        finally:\n            save_executor.shutdown(wait=True)\n            print(f"TIMING preview_images_total={image_seconds_total:.2f}s", flush=True)\n            print(f"TIMING preview_save_work_total={image_save_work_total:.2f}s", flush=True)\n            print(f"PREVIEW_PROCESS_TIME={time.monotonic() - started:.2f}s", flush=True)\n            if torch.cuda.is_available():\n                try:\n                    torch.cuda.synchronize()\n                except Exception as exc:\n                    print(f"GPU synchronize during preview cleanup: {type(exc).__name__}: {exc}", flush=True)\n            del pipe, transformer, vae\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n    if failures:\n        print(f"{len(failures)} preview card(s) failed; saved checkpoints remain valid.", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--cache", required=True)\n    parser.add_argument("--adapter", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--step", type=int, required=True)\n    parser.add_argument("--resolution", type=int, default=512)\n    args = parser.parse_args()\n    try:\n        main(args)\n    except Exception as exc:\n        manifest = Path(args.output) / "previews" / "latest.json"\n        mark_failed(manifest, exc)\n        print(f"TURBO_PREVIEW_FAILED: {type(exc).__name__}: {exc}", flush=True)\n        raise\n'

Path('/content/krea2_t4_precache.py').write_text(PRECACHE_SOURCE, encoding='utf-8')
Path('/content/krea2_t4_train.py').write_text(TRAIN_SOURCE, encoding='utf-8')
Path('/content/krea2_t4_preview.py').write_text(PREVIEW_SOURCE, encoding='utf-8')
py_compile.compile('/content/krea2_t4_precache.py', doraise=True)
py_compile.compile('/content/krea2_t4_train.py', doraise=True)
py_compile.compile('/content/krea2_t4_preview.py', doraise=True)
print('BACKEND_READY')


In [ ]:
import gc
import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs
from IPython.display import HTML, display
from PIL import Image
from huggingface_hub import HfApi
from safetensors import safe_open

PREVIEW_MODEL_REPO = "OzzyGT/Krea_2_Turbo_bnb_nf4"
PREVIEW_MODEL_REVISION = "5458debf8356a6646a5aa814de28dcea881f8a6d"
PREVIEW_MODEL_PATTERNS = ["model_index.json", "scheduler/*", "transformer/*"]
PREVIEW_MODEL_MIN_BYTES = 8 * 1024**3
PREVIEW_DISK_RESERVE_BYTES = 2 * 1024**3


def validate_preview_snapshot(snapshot):
    root = Path(snapshot)
    required = (root / "model_index.json", root / "scheduler" / "scheduler_config.json",
                root / "transformer" / "config.json")
    missing = [str(path.relative_to(root)) for path in required if not path.is_file()]
    weights = [path for path in (root / "transformer").rglob("*.safetensors")
               if path.is_file() and path.stat().st_size > 0]
    if missing or not weights:
        details = ", ".join(missing + ([] if weights else ["transformer/*.safetensors"]))
        raise RuntimeError(f"Turbo preview snapshot is incomplete: missing {details}.")
    if root.name != PREVIEW_MODEL_REVISION:
        raise RuntimeError(f"Turbo preview snapshot revision mismatch: expected {PREVIEW_MODEL_REVISION}, got {root.name}.")
    return root


def build_train_command(config, cache, output, resume=False):
    command = ["/content/krea2_t4_train.py", "--cache", str(cache), "--output", str(output),
               "--steps", str(config["steps"]), "--rank", str(config["rank"]),
               "--lr", str(config["lr"]), "--seed", str(config["seed"]),
               "--save-every", str(min(config["save_every"], config["steps"])),
               "--keep-checkpoints", str(config["keep"]), "--grad-accum", "1"]
    if config["grad_ckpt"]:
        command.append("--grad-ckpt")
    if config["attention_only"]:
        command.append("--attention-only")
    if config.get("preview_enabled"):
        command.append("--preview-at-checkpoints")
    if resume:
        command.append("--resume")
    return command


def build_preview_command(config, cache, adapter, output, preview_config, step):
    return ["/content/krea2_t4_preview.py", "--cache", str(cache),
            "--adapter", str(adapter), "--output", str(output),
            "--config", str(preview_config), "--step", str(int(step)),
            "--resolution", str(int(config["preview_resolution"]))]


class GpuMemoryReleaseError(RuntimeError):
    """A preview child kept enough VRAM allocated to endanger training resume."""


class Field:
    def __init__(self, value=None):
        self.value = value
        self.options = []
        self.disabled = False


def elapsed_label(seconds):
    seconds = max(0, int(seconds))
    return f"{seconds}s" if seconds < 60 else f"{seconds // 60}m {seconds % 60:02d}s"


class KreaStudio:
    """HTTP-backed UI state around the existing process-isolated training stages."""

    def __init__(self):
        self.closed = threading.Event()
        self.busy = False
        self.dataset_dir = None
        self.run_label = "Ready"
        self.log_lines = []
        self.last_progress = (-1, "")
        self.last_result = None
        self.timings = {"setup": globals().get("KREA_SETUP_SECONDS")}
        self.preview_prefetch_result = None
        self.stage_started = None
        self.current_stage = None
        self.cache_started = None
        self.vae_started = self.vae_first = self.qwen_started = self.qwen_first = None
        self.train_started = self.train_first = None
        self.last_step_at = None
        self.last_step = self.train_target = None
        self.sec_per_step = self.eta_seconds = None
        self.resource_updated_at = 0.0

        defaults = {"project":"my_krea2_lora", "trigger":"my_trigger_word", "caption":"character appearance and style",
            "use_txt":True, "test_mode":False, "resolution":512, "steps":500, "rank":16,
            "lr":0.0003, "save_every":50, "max_seq":128, "attention_only":True,
            "grad_ckpt":True, "seed":42, "preview_enabled":False,
            "preview_1_enabled":True, "preview_1_prompt":"", "preview_1_seed":42, "preview_1_random":False,
            "preview_2_enabled":False, "preview_2_prompt":"", "preview_2_seed":43, "preview_2_random":False,
            "preview_3_enabled":False, "preview_3_prompt":"", "preview_3_seed":44, "preview_3_random":False,
            "preview_4_enabled":False, "preview_4_prompt":"", "preview_4_seed":45, "preview_4_random":False,
            "preview_resolution":512, "preview_cleanup_cache":True,
            "push":False, "hub_id":"", "private":True, "keep":2, "drive":False, "token":""}
        for name, value in defaults.items():
            setattr(self, name, Field(value))
        self.dataset_path = Field("")
        for name in ("status", "resources", "progress", "dataset_summary", "results", "log"):
            setattr(self, name, Field(""))
        for name in ("upload_button", "path_button", "cache_button", "train_button", "resume_button", "sync_button"):
            setattr(self, name, Field())
        self._set_status("Ready", "Choose a dataset to begin.")
        self._set_progress(0, "Waiting for dataset")
        self._render_log()
        self._render_resources()
        self.refresh_results()

    def close(self):
        self.closed.set()

    def _root(self):
        name = re.sub(r"[^A-Za-z0-9._-]+", "_", self.project.value.strip()).strip("._")
        if not name:
            raise ValueError("Enter a project name.")
        if self.drive.value:
            from google.colab import drive
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
            return Path("/content/drive/MyDrive/Krea2_Colab") / name
        return Path("/content/Krea2_Colab") / name

    def _config(self):
        steps = min(self.steps.value, 20) if self.test_mode.value else self.steps.value
        rank = min(self.rank.value, 8) if self.test_mode.value else self.rank.value
        if not 0 < self.lr.value < .01:
            raise ValueError("Learning rate must be between 0 and 0.01.")
        if not self.trigger.value.strip():
            raise ValueError("Enter a trigger word.")
        save_every = max(1, min(int(self.save_every.value), int(steps)))
        preview_enabled = bool(self.preview_enabled.value)
        preview_resolution = int(self.preview_resolution.value)
        if preview_resolution not in (512, 768, 1024):
            raise ValueError("Turbo preview size must be 512, 768, or 1024 pixels.")
        preview_slots = []
        if preview_enabled:
            for slot in range(1, 5):
                if not getattr(self, f"preview_{slot}_enabled").value:
                    continue
                prompt = str(getattr(self, f"preview_{slot}_prompt").value).strip()
                if not prompt:
                    raise ValueError(f"Enter a prompt for Preview card {slot}, or turn that card off.")
                if len(prompt) > 2000:
                    raise ValueError(f"Preview card {slot} prompt must be 2,000 characters or fewer.")
                seed = int(getattr(self, f"preview_{slot}_seed").value)
                if not 0 <= seed <= 4294967295:
                    raise ValueError(f"Preview card {slot} seed must be between 0 and 4,294,967,295.")
                preview_slots.append({"slot":slot, "enabled":True, "prompt":prompt, "seed":seed,
                                      "random_seed":bool(getattr(self, f"preview_{slot}_random").value)})
            if not preview_slots:
                raise ValueError("Enable at least one Preview card, or turn preview generation off.")
        return {"steps": steps, "rank": rank, "resolution": self.resolution.value,
                "max_seq": self.max_seq.value, "lr": self.lr.value, "save_every": save_every,
                "seed": self.seed.value, "attention_only": self.attention_only.value,
                "grad_ckpt": self.grad_ckpt.value, "preview_enabled":preview_enabled,
                "preview_resolution":preview_resolution,
                "preview_cleanup_cache":bool(self.preview_cleanup_cache.value),
                "preview_slots":preview_slots,
                "push": self.push.value, "private": self.private.value, "keep": self.keep.value,
                "hub_id": self.hub_id.value.strip(), "drive": self.drive.value,
                "trigger": self.trigger.value.strip(), "caption": self.caption.value.strip(),
                "use_txt": self.use_txt.value}

    def _token(self):
        value = self.token.value.strip()
        if not value:
            try:
                from google.colab import userdata
                value = (userdata.get("HF_TOKEN") or "").strip()
            except Exception:
                pass
        if not value:
            raise ValueError("Set HF_TOKEN in Colab Secrets or enter a token above.")
        return value

    def _repo_id(self, token):
        if self.hub_id.value.strip():
            return self.hub_id.value.strip()
        username = HfApi(token=token).whoami()["name"]
        return f"{username}/{self._root().name}"

    def _set_status(self, title, detail=""):
        self.run_label = title
        self.status.value = ("<div style='border:2px solid #000;padding:16px;background:#fff;color:#000'>"
                             f"<b style='font-size:18px'>{html.escape(title)}</b><br>{html.escape(detail)}</div>")

    def _set_progress(self, pct, label):
        pct = max(0, min(100, int(pct)))
        if (pct, label) == self.last_progress:
            return
        self.last_progress = (pct, label)
        self.progress.value = ("<div style='padding:10px 0;color:#000'><b>Progress</b> · "
                               f"{html.escape(label)} · {pct}%<div style='height:14px;border:1px solid #000;background:#fff'>"
                               f"<div style='height:14px;width:{pct}%;background:#000'></div></div></div>")

    def _append_log(self, line):
        line = line.strip()
        if not line or line.startswith("Loading weights:") or line.startswith("Fetching"):
            return
        if line.startswith("/usr/local/lib/") or "FutureWarning" in line:
            return
        self.log_lines.append(line[:600])
        self.log_lines = self.log_lines[-80:]
        self._render_log()

    def _render_log(self):
        lines = "\n".join(self.log_lines[-24:]) or "Stage messages will appear here."
        self.log.value = ("<div style='border:1px solid #000;padding:12px;background:#000;color:#fff'>"
                          "<b>Activity</b><pre style='white-space:pre-wrap;max-height:260px;overflow:auto;color:#fff'>"
                          f"{html.escape(lines)}</pre></div>")

    def _render_resources(self):
        try:
            import psutil
            ram = psutil.virtual_memory()
            cpu = psutil.cpu_percent(interval=None)
            disk = shutil.disk_usage("/content")
            query = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=4)
            gpu = query.stdout.strip().splitlines()[0] if query.returncode == 0 and query.stdout.strip() else "unavailable"
            if gpu != "unavailable":
                used, total, util = [v.strip() for v in gpu.split(",")]
                gpu = f"{int(used)/1024:.1f} / {int(total)/1024:.1f} GB · {util}% busy"
            self.resource_updated_at = time.time()
            self.resources.value = ("<div><b>LIVE SYSTEM</b> · "
                f"CPU {cpu:.0f}% · RAM {ram.used/2**30:.1f}/{ram.total/2**30:.1f} GB"
                f" · GPU {html.escape(gpu)} · Free disk {disk.free/2**30:.1f} GB"
                f"<small style='display:block;color:#888;margin-top:4px'>Sampled {time.strftime('%H:%M:%S')}</small></div>")
        except Exception as e:
            self.resources.value = f"<div>System metrics unavailable: {html.escape(str(e))}</div>"

    def _resource_loop(self):
        while not self.closed.is_set():
            self._render_resources()
            self.closed.wait(2 if self.busy else 6)

    def _set_busy(self, busy):
        self.busy = busy
        for button in (self.upload_button, self.path_button, self.cache_button, self.train_button,
                       self.resume_button, self.sync_button):
            button.disabled = busy

    def _start(self, name, fn):
        if self.busy:
            return
        self._set_busy(True)
        self.current_stage = name
        self._set_status(name.title(), "Stage running. Keep this Colab tab open.")
        self._set_progress(0, "Starting")
        if name in ("train", "resume"):
            self.train_first = None
            self.last_step_at = None
            self.last_step = self.train_target = None
            self.sec_per_step = None
            self.eta_seconds = None
            self.timings.pop("train_cold_start", None)
            self.timings.pop("train_total", None)
            self.timings.pop("train_loop_total", None)
            self.timings.pop("preview_total", None)
        if name == "cache":
            self.vae_started = self.vae_first = self.qwen_started = self.qwen_first = None
            for key in ("vae_load", "vae_encode", "vae_total", "vae_stage", "qwen_load", "caption_encode", "cache_total"):
                self.timings.pop(key, None)
        try:
            fn()
            self._set_status("Complete", f"{name.title()} completed successfully.")
            self._set_progress(100, "Complete")
        except Exception as exc:
            self.last_result = exc
            self._append_log(f"ERROR: {type(exc).__name__}: {exc}")
            self._set_status("Needs attention", str(exc))
        finally:
            self._set_busy(False)
            self.refresh_results()
            self._render_resources()

    def _new_dataset_dir(self):
        path = self._root() / f"dataset_{int(time.time())}"
        path.mkdir(parents=True, exist_ok=False)
        return path

    def _ingest_files(self, files):
        allowed = {".zip", ".png", ".jpg", ".jpeg", ".webp", ".txt"}
        out = self._new_dataset_dir()
        count = 0
        for name, data in files:
            if Path(name).suffix.lower() not in allowed:
                continue
            if Path(name).suffix.lower() == ".zip":
                import io
                with zipfile.ZipFile(io.BytesIO(data)) as archive:
                    count += self._extract_archive(archive, out)
            else:
                target = out / Path(name).name
                if target.exists():
                    raise ValueError(f"Duplicate filename: {target.name}")
                target.write_bytes(data)
                count += 1
        self._validate_dataset(out)
        self.dataset_dir = out
        self._append_log(f"Dataset ready: {out} ({count} files)")
        self._set_status("Dataset ready", f"{len(self._images(out))} images. Next: Prepare cache.")

    def _extract_archive(self, archive, out):
        allowed = {".png", ".jpg", ".jpeg", ".webp", ".txt"}
        infos = archive.infolist()
        if len(infos) > 2000 or sum(i.file_size for i in infos) > 5 * 2**30:
            raise ValueError("ZIP is too large or contains too many entries.")
        count = 0
        for entry in infos:
            if entry.is_dir() or Path(entry.filename).suffix.lower() not in allowed:
                continue
            target = out / Path(entry.filename).name
            if target.exists():
                raise ValueError(f"Duplicate filename: {target.name}")
            with archive.open(entry) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)
            count += 1
        return count

    def _images(self, directory):
        return sorted(p for p in directory.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})

    def _validate_dataset(self, directory):
        images = self._images(directory)
        if not images:
            raise ValueError("No JPG, PNG or WEBP images found.")
        for image in images:
            with Image.open(image) as opened:
                opened.verify()
        captions = sum(p.with_suffix(".txt").exists() for p in images)
        self.dataset_summary.value = ("<div style='border:1px solid #000;padding:12px;color:#000'>"
            f"<b>Dataset</b> · {len(images)} valid images · {captions} matching captions<br>"
            f"<small>{html.escape(str(directory))}</small></div>")

    def _use_upload(self):
        if self.busy:
            return
        try:
            selected = self.upload.value
            items = list(selected.values()) if isinstance(selected, dict) else list(selected)
            if not items:
                raise ValueError("Choose a ZIP or image files first.")
            pairs = [(item["name"], bytes(item["content"])) for item in items]
            self._ingest_files(pairs)
            self.upload.value = ()
        except Exception as exc:
            self._set_status("Dataset error", str(exc))

    def _use_path(self):
        if self.busy:
            return
        try:
            path = Path(self.dataset_path.value.strip()).expanduser()
            if path.is_dir():
                self._validate_dataset(path)
                self.dataset_dir = path
                self._set_status("Dataset ready", f"{len(self._images(path))} images. Next: Prepare cache.")
            elif path.is_file() and path.suffix.lower() == ".zip":
                out = self._new_dataset_dir()
                with zipfile.ZipFile(path) as archive:
                    count = self._extract_archive(archive, out)
                self._validate_dataset(out)
                self.dataset_dir = out
                self._append_log(f"Dataset ready: {out} ({count} files)")
                self._set_status("Dataset ready", f"{len(self._images(out))} images. Next: Prepare cache.")
            else:
                raise ValueError("Enter an existing ZIP file or folder path in Colab.")
        except Exception as exc:
            self._set_status("Dataset error", str(exc))

    def _cache_signature(self, config):
        if self.dataset_dir is None:
            raise ValueError("Load a dataset first.")
        files = sorted(p for p in self.dataset_dir.iterdir() if p.suffix.lower() in
            {".png", ".jpg", ".jpeg", ".webp", ".txt"})
        payload = []
        for path in files:
            digest = hashlib.sha256()
            with path.open("rb") as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(chunk)
            payload.append((path.name, digest.hexdigest()))
        preview_prompts = sorted({slot["prompt"] for slot in config.get("preview_slots", [])})
        payload.append({"model":"krea/Krea-2-Raw", "cache_format":3,
                        **{k:config[k] for k in ("resolution", "max_seq", "trigger", "caption", "use_txt")},
                        "preview_prompts":preview_prompts})
        return hashlib.sha256(repr(payload).encode()).hexdigest()

    @staticmethod
    def _preview_embedding_paths(cache, prompt, max_seq):
        payload = f"caption-cache-v1\0krea/Krea-2-Raw\0{max_seq}\0{prompt}"
        key = hashlib.sha256(payload.encode("utf-8")).hexdigest()
        directory = Path(cache) / "caption_embeddings"
        return directory / f"{key}_embed.pt", directory / f"{key}_mask.pt"

    @staticmethod
    def _write_preview_config(config, path):
        payload = {"enabled":bool(config.get("preview_enabled")),
                   "slots":config.get("preview_slots", []) if config.get("preview_enabled") else []}
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + ".tmp")
        temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        os.replace(temporary, path)
        return path

    def _cache_complete(self, cache, signature):
        marker = cache / "studio_signature.json"
        metadata_file = cache / "metadata.json"
        try:
            if json.loads(marker.read_text(encoding="utf-8")).get("signature") != signature:
                return False
            metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
            items = metadata.get("items", [])
            if len(items) != len(self._images(self.dataset_dir)):
                return False
            for item in items:
                for key in ("latent", "embed", "mask"):
                    artifact = cache / item[key]
                    if not artifact.is_file() or artifact.stat().st_size == 0:
                        return False
            for slot in getattr(self, "_active_preview_slots", []):
                for artifact in self._preview_embedding_paths(cache, slot["prompt"], self.max_seq.value):
                    if not artifact.is_file() or artifact.stat().st_size == 0:
                        return False
            return True
        except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError):
            return False

    @staticmethod
    def _gpu_memory_snapshot():
        try:
            query = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=4)
            if query.returncode != 0 or not query.stdout.strip():
                return None
            values = [int(value.strip()) for value in query.stdout.strip().splitlines()[0].split(",")]
            return (values[0], values[1]) if len(values) == 2 else None
        except Exception:
            return None

    def _wait_for_gpu_release(self, before, timeout=15.0):
        if before is None:
            self.timings["preview_vram_reclaim"] = None
            self._append_log("VRAM release probe unavailable; the preview worker exited in its own process.")
            return
        started = time.monotonic()
        after = None
        threshold = before[0] + 512
        while True:
            after = self._gpu_memory_snapshot()
            if after is None or after[0] <= threshold or time.monotonic() - started >= timeout:
                break
            time.sleep(0.5)
        elapsed = time.monotonic() - started
        self.timings["preview_vram_reclaim"] = elapsed
        if after is None:
            self._append_log("VRAM release probe became unavailable after preview; process isolation completed.")
            return
        message = (f"Preview VRAM after process exit: {after[0] / 1024:.2f}/{after[1] / 1024:.2f} GiB; "
                   f"pre-preview baseline {before[0] / 1024:.2f} GiB; check {elapsed_label(elapsed)}.")
        self._append_log(message)
        if after[0] > threshold:
            raise GpuMemoryReleaseError(
                message + " Training remains at its saved checkpoint because VRAM did not return near baseline.")

    def _run_process(self, command, env, stage, total, allowed_return_codes=()):
        gpu_before = self._gpu_memory_snapshot() if stage == "preview" else None
        started = time.monotonic()
        self.stage_started = started
        self.current_stage = stage
        process = subprocess.Popen([sys.executable, "-u", *command], stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, env=env, bufsize=0)
        pending = ""
        next_resource_update = 0
        try:
            while True:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                if time.monotonic() >= next_resource_update:
                    self._render_resources()
                    next_resource_update = time.monotonic() + 2
                pending += chunk.decode("utf-8", errors="replace")
                parts = re.split(r"[\r\n]", pending)
                pending = parts.pop()
                for line in parts:
                    self._process_line(line, stage, total)
            if pending.strip():
                self._process_line(pending, stage, total)
            return_code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            process.stdout.close()
        if stage == "preview":
            self._wait_for_gpu_release(gpu_before)
        elapsed = time.monotonic() - started
        timing_key = f"{stage}_total"
        self.timings[timing_key] = self.timings.get(timing_key, 0.0) + elapsed
        self._append_log(f"{stage.title()} process time: {elapsed_label(elapsed)}.")
        if return_code and return_code not in allowed_return_codes:
            raise RuntimeError(f"{stage} exited with code {return_code}; see Activity.")
        return return_code

    def _process_line(self, line, stage, total):
        line = line.strip()
        if not line:
            return
        timing = re.match(r"^TIMING\s+([a-z0-9_]+)=([\d.]+)s$", line, re.I)
        if timing:
            key, value = timing.group(1).lower(), float(timing.group(2))
            self.timings[key] = value
            if key == "train_steady_sec_per_step":
                self.sec_per_step = value
                self.timings["sec_per_step"] = value
            if key == "caption_cache":
                self.timings["caption_encode"] = value
            if key in {"qwen_download", "tokenizer_load", "qwen_model_load"}:
                self.timings["qwen_load"] = sum(self.timings.get(k, 0) for k in
                    ("qwen_download", "tokenizer_load", "qwen_model_load"))
        if stage == "cache":
            now = time.monotonic()
            if "[1/2]" in line:
                self.vae_started = now
            if "[2/2]" in line:
                self.qwen_started = now
                if self.vae_started is not None:
                    self.timings["vae_stage"] = now - self.vae_started
            vae = re.search(r"\b(\d+)/(\d+)\s+\S+\.(?:png|jpg|jpeg|webp)\s+->", line, re.I)
            caption = re.search(r"caption\s+(\d+)/(\d+)", line, re.I)
            if vae:
                if self.vae_first is None:
                    self.vae_first = now
                    if self.vae_started is not None:
                        self.timings["vae_load"] = now - self.vae_started
                n, count = int(vae.group(1)), max(1, int(vae.group(2)))
                self._set_progress(5 + 45 * n / count, f"VAE encode · {n}/{count}")
                if n >= count and self.vae_first is not None:
                    self.timings["vae_encode"] = now - self.vae_first
            elif caption:
                if self.qwen_first is None:
                    self.qwen_first = now
                    if self.qwen_started is not None and "qwen_load" not in self.timings and "reused" not in line:
                        self.timings["qwen_load"] = now - self.qwen_started
                n, count = int(caption.group(1)), max(1, int(caption.group(2)))
                self._set_progress(55 + 45 * n / count, f"Caption cache · {n}/{count}")
                if n >= count and self.qwen_first is not None:
                    self.timings["caption_encode"] = now - self.qwen_first
            elif "[2/2]" in line:
                self._set_progress(55, "Loading Qwen3-VL in NF4")
        elif stage == "preview":
            sample = re.search(r"\bpreview\s+(\d+)/(\d+)\b", line, re.I)
            if sample:
                n, count = int(sample.group(1)), max(1, int(sample.group(2)))
                self._set_progress(100 * n / count, f"Turbo preview · {n}/{count}")
            if line.startswith("TIMING "):
                self._append_log(line)
                return
        elif stage == "train":
            now = time.monotonic()
            step = re.search(r"\bstep\s+(\d+)/(\d+)\b", line, re.I)
            speed = re.search(r"([\d.]+)s/step", line, re.I)
            eta = re.search(r"ETA\s+([\d.]+)\s*([smh])", line, re.I)
            if step:
                n, count = int(step.group(1)), max(1, int(step.group(2)))
                self.last_step_at = now
                self.last_step, self.train_target = n, count
                self._set_progress(100 * n / count, f"Training · {n}/{count}")
                if self.train_first is None:
                    self.train_first = now
                    if self.stage_started is not None:
                        self.timings["train_cold_start"] = now - self.stage_started
                if speed:
                    self.sec_per_step = float(speed.group(1))
                    self.timings["sec_per_step"] = self.sec_per_step
                if eta:
                    self.eta_seconds = float(eta.group(1)) * {"s":1,"m":60,"h":3600}[eta.group(2).lower()]
                elif self.sec_per_step is not None:
                    self.eta_seconds = max(0, (count-n) * self.sec_per_step)
            elif "Loading NF4" in line:
                self._set_progress(2, "Loading Krea 2 transformer in NF4")
        self._append_log(line)

    def _prepare_cache(self):
        config = self._config()
        self._active_preview_slots = config.get("preview_slots", []) if config.get("preview_enabled") else []
        signature = self._cache_signature(config)
        root = self._root()
        cache = root / "cache"
        cache.mkdir(parents=True, exist_ok=True)
        marker = cache / "studio_signature.json"
        preview_config = self._write_preview_config(config, root / "preview_config.json")
        token = self._token() if config["preview_enabled"] else None
        prefetch_thread = None
        if config["preview_enabled"]:
            self._append_log("Compact Turbo NF4 prefetch started alongside cache preparation.")
            prefetch_thread = threading.Thread(
                target=self._prefetch_preview_model,
                args=(token, config["preview_cleanup_cache"]),
                name="krea2-turbo-prefetch", daemon=True)
            prefetch_thread.start()
        if self._cache_complete(cache, signature):
            self.timings.update({"vae_load":0.0,"vae_encode":0.0,"vae_stage":0.0,
                                 "qwen_load":0.0,"caption_encode":0.0,"cache_total":0.0})
            self._append_log("Cache is current and all latent/caption files are present; preprocessing skipped.")
            self._append_log("Cache reuse timing · VAE 0s · Qwen 0s · total 0s.")
            if prefetch_thread is not None:
                prefetch_thread.join()
                self._require_preview_prefetch_ready()
            return
        env = os.environ.copy()
        if token:
            env["HF_TOKEN"] = token
        else:
            env["HF_TOKEN"] = self._token()
        self.cache_started = time.monotonic()
        command = ["/content/krea2_t4_precache.py", "--dataset", str(self.dataset_dir),
                   "--cache", str(cache), "--resolution", str(config["resolution"]),
                   "--max-seq", str(config["max_seq"]), "--trigger", config["trigger"],
                   "--default-caption", config["caption"], "--preview-config", str(preview_config)]
        if config["use_txt"]:
            command.append("--use-txt")
        try:
            self._run_process(command, env, "cache", len(self._images(self.dataset_dir)))
        finally:
            if prefetch_thread is not None:
                prefetch_thread.join()
        if prefetch_thread is not None:
            self._require_preview_prefetch_ready()
        metadata = json.loads((cache / "metadata.json").read_text(encoding="utf-8"))
        if len(metadata["items"]) != len(self._images(self.dataset_dir)):
            raise RuntimeError("Cache item count differs from dataset image count.")
        for item in metadata["items"]:
            if any(not (cache / item[key]).is_file() or (cache / item[key]).stat().st_size == 0
                   for key in ("latent", "embed", "mask")):
                raise RuntimeError(f"Cache artifact is missing or empty for {item.get('source', item.get('id'))}.")
        for slot in self._active_preview_slots:
            if any(not path.is_file() or path.stat().st_size == 0
                   for path in self._preview_embedding_paths(cache, slot["prompt"], config["max_seq"])):
                raise RuntimeError(f"Cached text embeddings are missing for preview card {slot['slot']}.")
        marker_tmp = marker.with_suffix(".json.tmp")
        marker_tmp.write_text(json.dumps({"signature": signature, "items": len(metadata["items"]),
                                          "verified_at": time.time()}, indent=2), encoding="utf-8")
        os.replace(marker_tmp, marker)
        self.timings["cache_total"] = time.monotonic() - self.cache_started
        self._append_log(f"Cache ready: {len(metadata['items'])} images.")
        self._append_log("Cache timing · " + " · ".join(
            f"{label}: {elapsed_label(self.timings[key])}" for label, key in
            (("VAE load", "vae_load"), ("VAE encode", "vae_encode"),
             ("Qwen load", "qwen_load"), ("caption encode", "caption_encode"),
             ("total", "cache_total")) if key in self.timings))

    def _prefetch_preview_model(self, token, replace_full_precision_cache):
        started = time.monotonic()
        self.preview_prefetch_result = {"ready": False, "error": "Turbo prefetch did not complete."}
        try:
            from huggingface_hub import HfApi, scan_cache_dir, snapshot_download

            cache_info = scan_cache_dir()
            local_snapshot = None
            try:
                candidate = snapshot_download(
                    repo_id=PREVIEW_MODEL_REPO, revision=PREVIEW_MODEL_REVISION,
                    allow_patterns=PREVIEW_MODEL_PATTERNS, token=token, local_files_only=True)
                validate_preview_snapshot(candidate)
                local_snapshot = str(candidate)
                self._append_log("Compact NF4 Turbo cache already exists and passed integrity checks.")
            except Exception as exc:
                self._append_log(f"No complete local Turbo snapshot found; checking exact download size ({type(exc).__name__}).")

            expected_bytes = None
            if local_snapshot is None:
                model_info = HfApi(token=token).model_info(
                    PREVIEW_MODEL_REPO, revision=PREVIEW_MODEL_REVISION, files_metadata=True)
                selected = [item for item in model_info.siblings
                            if item.rfilename == "model_index.json" or item.rfilename.startswith("scheduler/")
                            or item.rfilename.startswith("transformer/")]
                sizes = []
                missing_sizes = []
                for item in selected:
                    size = getattr(item, "size", None)
                    if size is None:
                        size = getattr(getattr(item, "lfs", None), "size", None)
                    if size is None:
                        missing_sizes.append(item.rfilename)
                    else:
                        sizes.append(int(size))
                if not selected or missing_sizes:
                    detail = ", ".join(missing_sizes[:3]) or "no matching files reported"
                    raise RuntimeError(f"Hugging Face did not provide exact sizes for the pinned Turbo files ({detail}).")
                expected_bytes = sum(sizes)
                required_free = max(PREVIEW_MODEL_MIN_BYTES, expected_bytes) + PREVIEW_DISK_RESERVE_BYTES
                free = shutil.disk_usage("/content").free

                if free < required_free and replace_full_precision_cache:
                    old_repo = next((repo for repo in cache_info.repos
                                     if repo.repo_id == "krea/Krea-2-Turbo"), None)
                    revisions = [revision.commit_hash for revision in getattr(old_repo, "revisions", [])]
                    if revisions:
                        cleanup = cache_info.delete_revisions(*revisions)
                        if cleanup.expected_freed_size:
                            self._append_log(
                                "Low disk space: replacing only the re-downloadable full-precision Turbo Hugging Face cache "
                                f"({cleanup.expected_freed_size / 1024**3:.1f} GiB) with compact NF4. "
                                "Dataset, cache, checkpoints and LoRA outputs are kept.")
                            cleanup.execute()
                            free = shutil.disk_usage("/content").free

                if free < required_free:
                    self._append_log(
                        f"Compact Turbo prefetch stopped: {free / 1024**3:.1f} GiB free; "
                        f"{required_free / 1024**3:.1f} GiB is required for the exact download plus reserve. "
                        "No project data was removed; free disk space and run Prepare cache again.")
                    self.preview_prefetch_result = {
                        "ready": False,
                        "error": f"Only {free / 1024**3:.1f} GiB disk space is free; {required_free / 1024**3:.1f} GiB is required.",
                    }
                    return

                self._append_log(
                    f"Downloading pinned compact NF4 Turbo files ({expected_bytes / 1024**3:.1f} GiB; "
                    "text encoder and VAE excluded) while cache preparation runs.")
                candidate = snapshot_download(
                    repo_id=PREVIEW_MODEL_REPO, revision=PREVIEW_MODEL_REVISION,
                    allow_patterns=PREVIEW_MODEL_PATTERNS, token=token)
                validate_preview_snapshot(candidate)
                local_snapshot = str(candidate)

            self._append_log(
                f"Compact Turbo cache verified in {elapsed_label(time.monotonic() - started)}; "
                f"free disk {shutil.disk_usage('/content').free / 1024**3:.1f} GiB.")
            self.preview_prefetch_result = {"ready": True, "snapshot": local_snapshot}
        except Exception as exc:
            self._append_log(f"Turbo prefetch did not finish: {type(exc).__name__}: {exc}")
            self.preview_prefetch_result = {"ready": False, "error": f"{type(exc).__name__}: {exc}"}
        finally:
            self.timings["preview_prefetch"] = time.monotonic() - started

    def _require_preview_prefetch_ready(self):
        result = getattr(self, "preview_prefetch_result", None) or {}
        if result.get("ready"):
            return
        detail = result.get("error", "No verified Turbo cache result was recorded.")
        raise RuntimeError(
            "Turbo preview is enabled, but its model cache is not ready, so training was not started. "
            f"{detail} Disable Optional Turbo preview images to train without previews, or correct the network/disk issue and run Prepare cache again."
        )

    def _preview_complete(self, output, step, slots):
        if not slots:
            return True
        manifest = Path(output) / "previews" / "latest.json"
        try:
            states = json.loads(manifest.read_text(encoding="utf-8")).get("slots", {})
        except (OSError, ValueError, json.JSONDecodeError):
            return False
        for slot in slots:
            state = states.get(str(slot["slot"]), {})
            image = Path(output) / "previews" / f"slot_{slot['slot']}" / "latest.png"
            if state.get("status") != "ready" or state.get("step") != int(step) or not image.is_file():
                return False
        return True

    def _run_preview(self, config, cache, output, preview_config, step, env):
        slots = config.get("preview_slots", []) if config.get("preview_enabled") else []
        if not slots:
            return
        adapter = Path(output) / "versions" / f"lora_step_{int(step):05d}.safetensors"
        if not adapter.is_file():
            checkpoint = Path(output) / f"checkpoint-{int(step)}" / "pytorch_lora_weights.safetensors"
            if checkpoint.is_file():
                adapter = checkpoint
            else:
                raise RuntimeError(f"No saved LoRA adapter found for preview step {step}.")
        command = build_preview_command(config, cache, adapter, output, preview_config, step)
        self._append_log(f"Turbo preview stage · step {step} · {len(slots)} enabled card(s).")
        self._run_process(command, env, "preview", len(slots))

    def _train(self, resume):
        config = self._config()
        root = self._root()
        cache = root / "cache"
        output = root / "output"
        manifest_path = root / "run_manifest.json"

        if self.dataset_dir is None and resume and manifest_path.is_file():
            try:
                saved_path = json.loads(manifest_path.read_text(encoding="utf-8")).get("dataset_path", "")
                if saved_path and Path(saved_path).is_dir():
                    self._validate_dataset(Path(saved_path))
                    self.dataset_dir = Path(saved_path)
                    self._append_log(f"Restored dataset selection from run manifest: {self.dataset_dir}")
            except (OSError, ValueError, json.JSONDecodeError):
                pass
        if self.dataset_dir is None:
            raise ValueError("Choose the dataset again, or use Drive storage so the saved run can restore it after a runtime reset.")

        marker = cache / "studio_signature.json"
        if not (cache / "metadata.json").exists() or not marker.exists():
            raise ValueError("Prepare the cache before training.")
        cache_marker = json.loads(marker.read_text(encoding="utf-8"))
        if cache_marker.get("signature") != self._cache_signature(config):
            raise ValueError("Dataset or cache settings changed. Prepare the cache again.")

        run_fields = ("rank", "lr", "seed", "resolution", "max_seq", "save_every", "attention_only", "grad_ckpt",
                      "trigger", "caption", "use_txt", "push", "private", "hub_id", "drive",
                      "preview_enabled", "preview_slots", "preview_resolution", "preview_cleanup_cache")
        run_settings = {key: config[key] for key in run_fields}
        run_manifest = {}
        if manifest_path.is_file():
            try:
                run_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            except (OSError, ValueError):
                run_manifest = {}
        if resume and run_manifest.get("config"):
            saved_config = run_manifest["config"]
            mismatches = [key for key in run_fields if key in saved_config and saved_config.get(key) != run_settings[key]]
            if config["preview_enabled"] and not saved_config.get("preview_enabled", False):
                mismatches.append("preview_enabled")
            if mismatches:
                raise ValueError("Resume settings differ from the saved run: " + ", ".join(sorted(set(mismatches))) +
                                 ". Keep the original settings or start a new project.")
        elif resume:
            self._append_log("Legacy run has no settings manifest; checking available checkpoint files.")

        checkpoints = []
        for candidate in output.glob("checkpoint-*"):
            try:
                step = int(candidate.name.split("-")[-1])
            except ValueError:
                continue
            if candidate.is_dir() and (candidate / "training_state.pt").is_file() and (candidate / "pytorch_lora_weights.safetensors").is_file():
                checkpoints.append((step, candidate))
        if resume and not checkpoints:
            raise ValueError("No complete local resume checkpoint was found. Enable Drive storage before training to keep full optimizer state across runtime resets.")
        latest_step = max(checkpoints)[0] if checkpoints else 0
        if resume and config["steps"] < latest_step:
            raise ValueError(f"The saved checkpoint is already at step {latest_step}; set Full run steps to at least that value.")

        config_file = output / "training_config.json"
        if resume and config_file.exists():
            previous = json.loads(config_file.read_text(encoding="utf-8"))
            if previous.get("rank") not in (None, config["rank"]):
                raise ValueError("Resume rank differs from saved run. Use Train for a new run.")

        output.mkdir(parents=True, exist_ok=True)
        preview_config = self._write_preview_config(config, root / "preview_config.json")
        run_manifest = {"schema":2, "status":"running", "project":root.name,
                        "dataset_path":str(self.dataset_dir), "cache_signature":cache_marker["signature"],
                        "config":run_settings, "target_steps":config["steps"],
                        "last_complete_step":latest_step, "updated_at":time.time()}
        manifest_tmp = manifest_path.with_suffix(".json.tmp")
        manifest_tmp.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
        os.replace(manifest_tmp, manifest_path)

        token = self._token()
        env = os.environ.copy()
        env["HF_TOKEN"] = token
        if resume and config["preview_enabled"] and latest_step:
            if not self._preview_complete(output, latest_step, config["preview_slots"]):
                try:
                    self._run_preview(config, cache, output, preview_config, latest_step, env)
                except GpuMemoryReleaseError:
                    raise
                except Exception as exc:
                    self._append_log(f"Turbo preview at resumed step {latest_step} failed; training will continue: {exc}")

        try:
            current_resume = bool(resume)
            while True:
                command = build_train_command(config, cache, output, current_resume)
                if config["push"]:
                    command += ["--push-to-hub", "--hub-model-id", self._repo_id(token)]
                    if config["private"]:
                        command.append("--private-repo")
                return_code = self._run_process(command, env, "train", config["steps"], allowed_return_codes=(75,))
                if return_code != 75:
                    break
                latest_checkpoint = max(
                    ((int(p.name.split("-")[-1]), p) for p in output.glob("checkpoint-*")
                     if p.is_dir() and (p / "training_state.pt").is_file()
                     and (p / "pytorch_lora_weights.safetensors").is_file()),
                    default=(0, None), key=lambda row:row[0])
                step, _checkpoint_path = latest_checkpoint
                if step <= 0:
                    raise RuntimeError("Training requested a preview but no complete resume checkpoint was found.")
                latest_step = step
                run_manifest.update({"last_complete_step":step, "updated_at":time.time()})
                manifest_tmp.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
                os.replace(manifest_tmp, manifest_path)
                try:
                    self._run_preview(config, cache, output, preview_config, step, env)
                except GpuMemoryReleaseError:
                    raise
                except Exception as exc:
                    self._append_log(f"Turbo preview at step {step} failed; saved training checkpoint is safe and training will continue: {exc}")
                current_resume = True

            final = output / "pytorch_lora_weights.safetensors"
            if not final.is_file() or final.stat().st_size < 1024:
                raise RuntimeError("Training exited without a valid final LoRA file.")
            with safe_open(str(final), framework="pt", device="cpu") as f:
                tensor_count = len(f.keys())
            if tensor_count == 0:
                raise RuntimeError("Final LoRA has no tensors.")

            if config["preview_enabled"] and not self._preview_complete(output, config["steps"], config["preview_slots"]):
                try:
                    self._run_preview(config, cache, output, preview_config, config["steps"], env)
                except Exception as exc:
                    self._append_log(f"Final Turbo preview failed; the trained LoRA is still ready: {exc}")
        except Exception:
            completed = latest_step
            for candidate in output.glob("checkpoint-*"):
                try:
                    step = int(candidate.name.split("-")[-1])
                except ValueError:
                    continue
                if (candidate / "training_state.pt").is_file() and (candidate / "pytorch_lora_weights.safetensors").is_file():
                    completed = max(completed, step)
            run_manifest.update({"status":"interrupted", "last_complete_step":completed, "updated_at":time.time()})
            manifest_tmp.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
            os.replace(manifest_tmp, manifest_path)
            raise

        run_manifest.update({"status":"complete", "last_complete_step":config["steps"], "updated_at":time.time()})
        manifest_tmp.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
        os.replace(manifest_tmp, manifest_path)
        self._append_log(f"Final LoRA verified: {final.name}, {tensor_count} tensors, {final.stat().st_size/2**20:.1f} MB.")
        self._append_log("Training timing · " + " · ".join(
            f"{label}: {elapsed_label(self.timings[key])}" for label, key in
            (("cold start", "train_cold_start"), ("train total", "train_total"),
             ("Turbo previews", "preview_total")) if key in self.timings) +
            (f" · {self.sec_per_step:.2f}s/step" if self.sec_per_step is not None else ""))

    def _sync(self):
        if not self.push.value:
            raise ValueError("Enable Hub upload first.")
        output = self._root() / "output"
        versions = sorted((output / "versions").glob("lora_step_*.safetensors"))
        if not versions:
            raise ValueError("No LoRA versions to sync.")
        token = self._token()
        repo_id = self._repo_id(token)
        api = HfApi(token=token)
        api.create_repo(repo_id=repo_id, repo_type="model", private=self.private.value, exist_ok=True)
        files = versions + [p for p in (output / "pytorch_lora_weights.safetensors",
                                             output / "training_config.json") if p.exists()]
        for index, path in enumerate(files, 1):
            repo_path = f"checkpoints/{path.name}" if path in versions else path.name
            api.upload_file(path_or_fileobj=str(path), path_in_repo=repo_path,
                            repo_id=repo_id, repo_type="model")
            self._set_progress(index / len(files) * 100, f"Hub upload · {index}/{len(files)}")
        remote = {f.rfilename for f in api.repo_info(repo_id=repo_id, repo_type="model").siblings}
        if not all((f"checkpoints/{p.name}" if p in versions else p.name) in remote for p in files):
            raise RuntimeError("Hub verification did not find every uploaded file.")
        self._append_log(f"Hub verified: https://huggingface.co/{repo_id}")

    def list_loras(self):
        """Return only complete, non-empty adapter files in download-friendly order."""
        output = self._root() / "output"
        versions = []
        for path in (output / "versions").glob("lora_step_*.safetensors"):
            match = re.fullmatch(r"lora_step_(\d+)\.safetensors", path.name)
            if not match or not path.is_file():
                continue
            size = path.stat().st_size
            if size <= 0:
                continue
            step = int(match.group(1))
            versions.append({"filename":path.name, "kind":"checkpoint", "step":step,
                             "label":f"Checkpoint · {step:,} steps", "size_bytes":size,
                             "size_label":self._format_file_size(size)})
        versions.sort(key=lambda item: item["step"], reverse=True)

        final = output / "pytorch_lora_weights.safetensors"
        final_item = None
        if final.is_file() and final.stat().st_size > 0:
            step = None
            config_path = output / "training_config.json"
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
                step = int(config.get("steps")) if config.get("steps") is not None else None
            except (OSError, ValueError, TypeError, json.JSONDecodeError):
                pass
            if step is None and versions:
                step = versions[0]["step"]
            size = final.stat().st_size
            label = f"Final · {step:,} steps" if step is not None else "Final LoRA"
            final_item = {"filename":final.name, "kind":"final", "step":step, "label":label,
                          "size_bytes":size, "size_label":self._format_file_size(size)}
        return ([final_item] if final_item else []) + versions

    @staticmethod
    def _format_file_size(size):
        return f"{size / 2**30:.2f} GB" if size >= 2**30 else f"{size / 2**20:.1f} MB"

    def resolve_lora_file(self, filename):
        """Resolve a listed adapter by its exact basename; reject arbitrary paths."""
        if not isinstance(filename, str) or not filename or Path(filename).name != filename:
            raise ValueError("Choose a LoRA file from the saved-files list.")
        match = next((item for item in self.list_loras() if item["filename"] == filename), None)
        if match is None:
            raise FileNotFoundError("That LoRA is no longer available. Refresh the list and try again.")
        path = (self._root() / "output" / "versions" / filename if match["kind"] == "checkpoint"
                else self._root() / "output" / filename)
        if not path.is_file() or path.stat().st_size <= 0:
            raise FileNotFoundError("The selected LoRA is missing or incomplete. Refresh the list and try again.")
        return path

    def refresh_results(self):
        try:
            output = self._root() / "output"
            items = self.list_loras()
            self._lora_inventory = items
            count = len(items)
            final_ready = any(item["kind"] == "final" for item in items)
            self.results.value = ("<div style='color:#000'>"
                f"<b>Results</b> · {count} LoRA file(s) · Final LoRA: {'ready' if final_ready else 'pending'}"
                f"<br><small>{html.escape(str(output))}</small></div>")
        except Exception as exc:
            self._lora_inventory = []
            self.results.value = f"Results unavailable: {html.escape(str(exc))}"

    def timing_html(self):
        live = dict(self.timings)
        now = time.monotonic()
        live_eta = self.eta_seconds
        if self.busy and self.current_stage in ("train", "resume") and self.stage_started is not None:
            if self.train_first is None:
                live["train_cold_start"] = now - self.stage_started
            else:
                live["train_total"] = now - self.stage_started
            if self.eta_seconds is not None and self.last_step_at is not None:
                live_eta = max(0.0, self.eta_seconds - (now - self.last_step_at))
        if self.busy and self.current_stage == "cache" and self.cache_started is not None:
            live["cache_total"] = now - self.cache_started
        if self.busy and self.current_stage == "preview" and self.stage_started is not None:
            live["preview_total"] = live.get("preview_total", 0.0) + now - self.stage_started
        if self.busy and self.current_stage == "cache" and self.vae_first is not None and self.vae_started is not None and self.qwen_started is None:
            live["vae_encode"] = now - self.vae_first
        if self.busy and self.current_stage == "cache" and self.qwen_first is not None:
            live["caption_encode"] = now - self.qwen_first
        def pretty(value):
            if value is None:
                return "—"
            seconds = max(0, int(value))
            return f"{seconds}s" if seconds < 60 else f"{seconds//60}m {seconds%60:02d}s"
        values = [
            ("Setup", live.get("setup")), ("VAE load", live.get("vae_load")),
            ("VAE download", live.get("vae_download")), ("VAE + encode", live.get("vae_stage", live.get("vae_encode"))),
            ("Qwen download", live.get("qwen_download")), ("Tokenizer load", live.get("tokenizer_load")),
            ("Qwen NF4 load", live.get("qwen_model_load")), ("Qwen total", live.get("qwen_load")),
            ("Caption cache", live.get("caption_encode")), ("Cache total", live.get("cache_total")),
            ("Transformer download", live.get("transformer_download")),
            ("Transformer NF4 load", live.get("transformer_nf4_load")),
            ("Trainer setup", live.get("adapter_optimizer_setup")),
            ("Cached data to RAM", live.get("cache_ram_load")),
            ("First optimizer step", live.get("first_step_compute")),
            ("Train cold start", live.get("train_cold_start")),
            ("Train loop", live.get("train_loop_total")),
            ("Seconds / step", live.get("train_steady_sec_per_step", self.sec_per_step)),
            ("Train ETA · live", live_eta), ("Turbo prefetch", live.get("preview_prefetch")),
            ("Turbo download", live.get("preview_download")), ("Turbo NF4 load", live.get("preview_nf4_load")),
            ("Turbo VAE load", live.get("preview_vae_load")), ("Turbo pipeline", live.get("preview_pipeline_load")),
            ("Turbo LoRA attach", live.get("preview_adapter_load")),
            ("Turbo model load", live.get("preview_model_load")), ("Turbo image time", live.get("preview_images_total")),
            ("VRAM return wait", live.get("preview_vram_reclaim")),
            ("Turbo previews", live.get("preview_total")),
            ("Checkpoint save", live.get("checkpoint_local")),
            ("Checkpoint upload", live.get("checkpoint_upload")),
            ("Final export", live.get("final_export_save")), ("Final upload", live.get("final_upload")),
        ]
        cards = "".join(f"<div class='metric'><small>{html.escape(label)}</small><b>{pretty(value)}</b></div>" for label, value in values)
        return f"<div class='card'><h3>Timing</h3><div class='metric-grid'>{cards}</div></div>"

    def state(self):
        self.refresh_results()
        current_status = self.status.value
        if self.busy and self.stage_started is not None:
            elapsed = int(time.monotonic() - self.stage_started)
            current_status = current_status.replace("</div>", f"<div style='color:#777;margin-top:5px'>Current process · {elapsed//60}m {elapsed%60:02d}s elapsed</div></div>", 1)
        return {"status":current_status, "progress":self.progress.value, "system":self.resources.value,
                "timing":self.timing_html(), "logs":self.log.value, "results":self.results.value,
                "loras":getattr(self, "_lora_inventory", []),
                "dataset":self.dataset_summary.value, "previews":self.preview_state(), "busy":bool(self.busy)}

    def preview_state(self):
        manifest = self._root() / "output" / "previews" / "latest.json"
        try:
            value = json.loads(manifest.read_text(encoding="utf-8"))
            return value.get("slots", {})
        except (OSError, ValueError, json.JSONDecodeError):
            return {}


PAGE = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Krea 2 · LoRA Studio</title><style>
:root{color-scheme:light;--ink:#171717;--muted:#777;--line:#e8e8e8;--wash:#f7f7f6;--white:#fff}
*{box-sizing:border-box}body{margin:0;background:#fff;color:var(--ink);font:14px/1.5 Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif}
.wrap{max-width:1280px;margin:auto;padding:30px 32px 48px}.head{display:flex;align-items:center;justify-content:space-between;gap:18px;border-bottom:1px solid var(--line);padding:3px 0 20px;margin-bottom:18px}
h1{font-size:28px;letter-spacing:-1.2px;line-height:1.15;margin:0;font-weight:680}.sub{color:var(--muted);margin-top:6px}.tag{display:inline-flex;align-items:center;gap:8px;border:1px solid var(--line);border-radius:99px;padding:7px 11px;color:#555;font-size:12px;white-space:nowrap}.dot{width:7px;height:7px;border-radius:50%;background:#888}.tag[data-state="online"] .dot{background:#26805a}.tag[data-state="reconnecting"] .dot{background:#b7791f}.tag[data-state="offline"] .dot{background:#a33}
.steps{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:14px}.step,.card{border:1px solid var(--line);border-radius:12px;background:#fff;box-shadow:0 2px 12px #00000005}.step{padding:12px 15px;display:flex;align-items:center;gap:10px}.num{display:grid;place-items:center;width:25px;height:25px;border-radius:50%;background:#f3f3f2;color:#555;font-size:11px;font-weight:650}.step strong{font-size:13px;font-weight:600}
.layout{display:grid;grid-template-columns:minmax(330px,.92fr) minmax(440px,1.08fr);gap:14px;align-items:start}.layout>section:last-child{grid-column:1/-1;display:grid;grid-template-columns:minmax(0,1.2fr) minmax(0,.8fr);gap:14px;align-items:start}.card{padding:18px;margin-bottom:14px}h2{font-size:16px;letter-spacing:-.25px;margin:0 0 3px}h3{font-size:13px;margin:0 0 9px}.note{font-size:12px;color:var(--muted);margin:0 0 14px}.field{margin:10px 0}.field label{display:block;font-size:12px;font-weight:600;margin-bottom:5px}.field input,.field select,.field textarea{width:100%;border:1px solid #dedede;border-radius:7px;background:#fff;padding:7px 10px;color:var(--ink);font:inherit;outline:none}.field input,.field select{height:37px}.field textarea{min-height:74px;resize:vertical}.field input:focus,.field select:focus,.field textarea:focus{border-color:#777;box-shadow:0 0 0 2px #0000000c}.check{display:flex;align-items:center;gap:8px;margin:9px 0;font-size:12px;color:#333}.check input{accent-color:#171717}.subtle{padding:10px 12px;background:var(--wash);border-radius:8px;margin-top:12px}
button{font-weight:600;font-size:12px;font-family:inherit;cursor:pointer;border-radius:7px;padding:10px 12px;transition:background .15s,border .15s}.primary{border:1px solid var(--ink);background:var(--ink);color:white}.primary:hover{background:#333}.secondary{border:1px solid #dedede;background:white;color:var(--ink)}.secondary:hover{background:#f5f5f5}.buttons{display:flex;gap:8px;flex-wrap:wrap;margin-top:9px}.buttons button{flex:1}.library-head{display:flex;align-items:flex-start;justify-content:space-between;gap:12px}.library-head .note{margin:4px 0 0}.library-actions{display:flex;gap:7px;flex-shrink:0}.lora-list{display:grid;gap:6px;margin-top:10px;max-height:390px;overflow:auto}.lora-row{display:flex;align-items:center;justify-content:space-between;gap:12px;padding:9px 10px;border:1px solid var(--line);border-radius:5px;background:#fff}.lora-info{min-width:0}.lora-info strong{display:block;font-size:12px;font-weight:650}.lora-info span{display:block;color:var(--muted);font-size:11px;overflow-wrap:anywhere}.lora-row button{flex:0 0 auto;padding:8px 12px}.lora-empty{padding:14px;border:1px dashed #d8d8d8;border-radius:5px;color:var(--muted);font-size:12px;text-align:center}.library-head button{white-space:nowrap}.wide{width:100%;margin-top:8px}.disabled,button:disabled{opacity:.48;cursor:wait}.status,.meter{border:1px solid var(--line);border-radius:9px;padding:11px 12px;background:#fff;margin:9px 0}.meter{font-size:12px}.bar{height:7px;background:#eee;border-radius:9px;overflow:hidden;margin-top:8px}.bar i{display:block;height:100%;background:#171717;width:0;transition:width .25s}.monitor{display:grid;grid-template-columns:1fr 1fr;gap:9px}.monitor .status,.monitor .meter{margin:0;min-width:0}.metric-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}.metric{padding:9px;background:var(--wash);border-radius:7px;min-width:0}.metric small{display:block;color:#777;font-size:10px;letter-spacing:.35px}.metric b{display:block;margin-top:2px;font-size:14px;overflow-wrap:anywhere}.console{background:#171717;color:#eee;border-radius:9px;padding:13px}.console pre{white-space:pre-wrap;overflow:auto;max-height:300px;margin:8px 0 0;font:11px/1.6 ui-monospace,SFMono-Regular,Consolas,monospace}.hint{font-size:11px;color:#888}.drop{border:1px dashed #d8d8d8;border-radius:8px;padding:12px;background:#fafafa}.upload-progress{height:6px;background:#eee;border-radius:8px;margin-top:9px;overflow:hidden}.upload-progress i{display:block;height:100%;width:0;background:#111}.dataset{font-size:12px;color:#555;overflow-wrap:anywhere}.section-gap{margin-top:17px}.message{min-height:18px;color:#777;font-size:11px;margin-top:6px}.err{color:#9b1c1c}details summary{cursor:pointer;font-size:12px;font-weight:600;color:#444}details[open] summary{margin-bottom:9px}.row{display:grid;grid-template-columns:1fr 1fr;gap:10px}.preview-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px;margin-top:12px}.preview-card{border:1px solid var(--line);border-radius:9px;padding:10px;min-width:0}.preview-card .check{margin-top:0}.preview-frame{position:relative;display:grid;place-items:center;width:100%;aspect-ratio:4/3;background:#f5f5f4;border-radius:7px;overflow:hidden;color:#888;font-size:11px;text-align:center}.preview-frame img{width:100%;height:100%;object-fit:contain}.preview-placeholder{padding:12px}.preview-meta{min-height:18px;margin-top:6px}.preview-note{padding:10px 11px;background:var(--wash);border-radius:7px;margin:10px 0;font-size:11px;color:#666}.preview-estimate{font-size:11px;font-weight:600;margin-top:8px;color:#333}.preview-card .field{margin:8px 0}.divider{height:1px;background:var(--line);margin:14px 0}
@media(max-width:850px){.wrap{padding:22px}.layout{grid-template-columns:1fr 1fr}.layout>section:last-child{grid-column:1/-1}}@media(max-width:620px){.wrap{padding:14px}.head{align-items:flex-start;flex-direction:column;gap:11px}.steps{gap:6px}.step{padding:9px;gap:7px}.step strong{font-size:11px}.layout{display:block}.layout>section:last-child{grid-column:auto;display:block}.monitor{grid-template-columns:1fr}.row,.preview-grid{grid-template-columns:1fr}.library-head{display:block}.library-actions{margin-top:10px}.library-actions button{flex:1}.lora-row{align-items:flex-start}}
</style></head><body><main class="wrap">
<header class="head"><div><h1>Krea 2 <span style="font-weight:400">LoRA Studio</span></h1><div class="sub">A simple, staged workflow for training and saving a LoRA.</div></div><div class="tag" id="connection" data-state="connecting" role="status" aria-live="polite"><span class="dot"></span><span id="connection-label">Connecting to Colab…</span></div></header>
<div class="steps" aria-label="Workflow"><div class="step"><span class="num">1</span><strong>Configure</strong></div><div class="step"><span class="num">2</span><strong>Add dataset</strong></div><div class="step"><span class="num">3</span><strong>Prepare & train</strong></div></div>
<div class="layout"><section>
<div class="card"><h2>LoRA settings</h2><p class="note">Start with these settings or adjust the advanced options.</p>
<div class="field"><label for="project">Project name</label><input id="project" value="my_krea2_lora" autocomplete="off"></div>
<div class="field"><label for="trigger">Trigger word</label><input id="trigger" value="my_trigger_word" autocomplete="off"></div>
<div class="field"><label for="caption">Fallback caption</label><input id="caption" value="character appearance and style" autocomplete="off"></div>
<label class="check"><input id="use_txt" type="checkbox" checked>Use matching .txt captions when available</label>
<div class="subtle"><label class="check"><input id="test_mode" type="checkbox">Quick run · 20 steps, rank 8</label><div class="hint">Off by default. When enabled, Train runs 20 steps at rank 8.</div></div>
<div class="row"><div class="field"><label for="resolution">Resolution</label><select id="resolution"><option>512</option><option>640</option><option>768</option></select></div><div class="field"><label for="seed">Seed</label><input id="seed" type="number" value="42"></div></div>
<details class="section-gap"><summary>Advanced training settings</summary><div class="row"><div class="field"><label for="steps">Training steps</label><input id="steps" type="number" min="1" max="100000" value="500"></div><div class="field"><label for="rank">LoRA rank</label><select id="rank"><option>4</option><option>8</option><option selected>16</option><option>32</option></select></div></div>
<div class="row"><div class="field"><label for="lr">Learning rate</label><input id="lr" type="number" step="0.00001" value="0.0003"></div><div class="field"><label for="save_every">Save every steps</label><input id="save_every" type="number" min="1" value="50"></div></div>
<div class="row"><div class="field"><label for="max_seq">Caption token limit</label><select id="max_seq"><option>96</option><option selected>128</option><option>192</option><option>256</option></select></div><div class="field"><label for="keep">Resume versions to keep</label><input id="keep" type="number" min="1" max="20" value="2"></div></div>
<label class="check"><input id="attention_only" type="checkbox" checked>Attention-only LoRA</label><label class="check"><input id="grad_ckpt" type="checkbox" checked>Gradient checkpointing · lower VRAM use</label></details>
<details class="section-gap" id="preview-options"><summary>Optional Turbo preview images</summary>
<label class="check"><input id="preview_enabled" type="checkbox">Generate images at each saved LoRA checkpoint</label>
<p class="hint">Uses a compact, pre-quantized NF4 Krea 2 Turbo model at 8 steps. Preview resolution defaults to 512 px. Prompt embeddings are reused from Prepare cache; only the transformer, scheduler and VAE are loaded for preview.</p>
<div id="preview-settings" hidden>
<div class="row"><div class="field"><label for="preview_resolution">Preview size</label><select id="preview_resolution"><option selected>512</option><option>768</option><option>1024</option></select></div><div class="field"><label>Disk space</label><div class="hint subtle">Downloads alongside Prepare cache. If needed, only the full-precision Turbo Hugging Face cache can be replaced; dataset, cache, checkpoints and LoRA files stay.</div></div></div>
<label class="check"><input id="preview_cleanup_cache" type="checkbox" checked>Replace cached full-precision Turbo weights only if disk space is tight</label>
<div id="preview-estimate" class="preview-estimate">Configure preview cards to see the image count.</div>
<div class="preview-note">Each enabled card keeps its own prompt and seed. Static seeds repeat the same sampling setup; random seeds change at each checkpoint. Training resumes from a full checkpoint even if preview generation fails.</div>
<div class="preview-grid">
<article class="preview-card"><label class="check"><input id="preview_1_enabled" type="checkbox" checked>Preview 1</label><div class="preview-frame"><img id="preview-image-1" alt="Turbo preview 1" hidden><div id="preview-placeholder-1" class="preview-placeholder">Your first preview will appear here.</div></div><div id="preview-meta-1" class="hint preview-meta">No image generated yet.</div><div class="field"><label for="preview_1_prompt">Prompt</label><textarea id="preview_1_prompt" maxlength="2000" placeholder="Describe the image to sample"></textarea></div><div class="field"><label for="preview_1_seed">Seed</label><input id="preview_1_seed" type="number" min="0" max="4294967295" value="42"></div><label class="check"><input id="preview_1_random" type="checkbox">Random seed each checkpoint</label></article>
<article class="preview-card"><label class="check"><input id="preview_2_enabled" type="checkbox">Preview 2</label><div class="preview-frame"><img id="preview-image-2" alt="Turbo preview 2" hidden><div id="preview-placeholder-2" class="preview-placeholder">Your second preview will appear here.</div></div><div id="preview-meta-2" class="hint preview-meta">No image generated yet.</div><div class="field"><label for="preview_2_prompt">Prompt</label><textarea id="preview_2_prompt" maxlength="2000" placeholder="Describe the image to sample"></textarea></div><div class="field"><label for="preview_2_seed">Seed</label><input id="preview_2_seed" type="number" min="0" max="4294967295" value="43"></div><label class="check"><input id="preview_2_random" type="checkbox">Random seed each checkpoint</label></article>
<article class="preview-card"><label class="check"><input id="preview_3_enabled" type="checkbox">Preview 3</label><div class="preview-frame"><img id="preview-image-3" alt="Turbo preview 3" hidden><div id="preview-placeholder-3" class="preview-placeholder">Your third preview will appear here.</div></div><div id="preview-meta-3" class="hint preview-meta">No image generated yet.</div><div class="field"><label for="preview_3_prompt">Prompt</label><textarea id="preview_3_prompt" maxlength="2000" placeholder="Describe the image to sample"></textarea></div><div class="field"><label for="preview_3_seed">Seed</label><input id="preview_3_seed" type="number" min="0" max="4294967295" value="44"></div><label class="check"><input id="preview_3_random" type="checkbox">Random seed each checkpoint</label></article>
<article class="preview-card"><label class="check"><input id="preview_4_enabled" type="checkbox">Preview 4</label><div class="preview-frame"><img id="preview-image-4" alt="Turbo preview 4" hidden><div id="preview-placeholder-4" class="preview-placeholder">Your fourth preview will appear here.</div></div><div id="preview-meta-4" class="hint preview-meta">No image generated yet.</div><div class="field"><label for="preview_4_prompt">Prompt</label><textarea id="preview_4_prompt" maxlength="2000" placeholder="Describe the image to sample"></textarea></div><div class="field"><label for="preview_4_seed">Seed</label><input id="preview_4_seed" type="number" min="0" max="4294967295" value="45"></div><label class="check"><input id="preview_4_random" type="checkbox">Random seed each checkpoint</label></article>
</div></div></details>
<details class="section-gap"><summary>Optional Hugging Face upload</summary><label class="check"><input id="push" type="checkbox">Upload saved LoRA versions to Hugging Face</label><label class="check"><input id="private" type="checkbox" checked>Keep repository private</label><div class="field"><label for="hub_id">Repository ID</label><input id="hub_id" placeholder="username/model-name"></div><p class="hint">Add an HF_TOKEN secret in Colab and enable notebook access before starting. Leave upload off to train and download locally.</p></details>
<details class="section-gap"><summary>Google Drive storage</summary><label class="check"><input id="drive" type="checkbox">Keep dataset, cache & resume state on Drive</label><p class="hint">Enable before adding the dataset. Drive can preserve reusable cache and resume files after a runtime reset.</p></details>
</div></section>
<section><div class="card"><h2>Dataset</h2><p class="note">Upload one ZIP with images and optional matching .txt captions.</p><div class="drop"><input id="zip" type="file" accept=".zip,application/zip"><div class="upload-progress"><i id="uploadbar"></i></div><div id="uploadlabel" class="hint">A ZIP can be uploaded in retryable chunks.</div></div><button id="uploadbtn" class="secondary wide" onclick="uploadZip()">Upload dataset</button>
<div class="divider"></div><div class="field"><label for="dataset_path">Or choose a folder already in Colab or Drive</label><input id="dataset_path" placeholder="/content/my_dataset"></div><button class="secondary wide" onclick="usePath()">Use dataset folder</button><div id="dataset" class="dataset section-gap">No dataset selected yet.</div></div>
<div class="card"><h2>Training</h2><p class="note">Each model stage exits before the next large model loads.</p><div class="hint">1 · Build reusable VAE and caption cache<br>2 · Load Krea 2 and train the adapter<br>3 · Save versions and final LoRA</div>
<button class="primary wide job" onclick="runAction('cache')">Prepare cache</button><div class="buttons"><button class="primary job" onclick="runAction('train')">Train</button><button class="secondary job" onclick="runAction('resume')">Resume</button></div>
<button class="secondary wide job" onclick="runAction('sync')">Upload saved versions</button><div id="action-message" class="message" role="status" aria-live="polite"></div></div>
<div class="card" id="lora-library"><div class="library-head"><div><h2>Saved LoRA files</h2><p class="note">Download a checkpoint or the final model. Transfers start directly and can run while training continues.</p></div><div class="library-actions"><button class="secondary" id="lora-refresh" type="button">Refresh</button><button class="primary" id="lora-download-all" type="button" disabled>Download all</button></div></div><div id="lora-summary" class="hint">No saved LoRA yet.</div><div id="lora-list" class="lora-list" aria-live="polite"><div class="lora-empty">Saved LoRA files will appear here.</div></div><div id="lora-message" class="message" role="status" aria-live="polite"></div></div></section>
<section><div class="card"><h2>Run monitor</h2><p class="note">Progress, live system readings and stage timings.</p><div class="monitor"><div id="status" class="status">Starting…</div><div id="progress" class="meter">Waiting for dataset<div class="bar"><i></i></div></div><div id="system" class="meter">Reading system metrics…</div><div id="timing" class="meter">Stage timings will appear here.</div></div></div><div class="console"><strong>Activity</strong><pre id="logs">Stage output will appear here.</pre></div></section></div></main>
<script>
const KEY=decodeURIComponent(location.hash.slice(1));let busy=false;const $=id=>document.getElementById(id);
function config(){const c={};['project','trigger','caption','resolution','preview_resolution','steps','rank','lr','save_every','max_seq','seed','keep','hub_id'].forEach(k=>c[k]=$(k).value);['use_txt','test_mode','attention_only','grad_ckpt','push','private','drive','preview_enabled','preview_cleanup_cache'].forEach(k=>c[k]=$(k).checked);for(let i=1;i<=4;i++){c['preview_'+i+'_prompt']=$('preview_'+i+'_prompt').value;c['preview_'+i+'_seed']=$('preview_'+i+'_seed').value;c['preview_'+i+'_enabled']=$('preview_'+i+'_enabled').checked;c['preview_'+i+'_random']=$('preview_'+i+'_random').checked}return c}
function updatePreviewEstimate(){const c=config(),active=[];for(let i=1;i<=4;i++)if(c['preview_'+i+'_enabled'])active.push(i);const steps=c.test_mode?Math.min(Number(c.steps)||20,20):(Number(c.steps)||500),save=Math.max(1,Number(c.save_every)||50),intervals=Math.ceil(steps/save);$('preview-estimate').textContent=!c.preview_enabled?'Preview is off · training speed is unchanged.':active.length?`${active.length} card(s) × ${intervals} checkpoints = ${active.length*intervals} planned image(s) · ${intervals} Turbo handoff(s).`:'Enable at least one card to generate previews.'}
function updatePreviewControls(){ $('preview-settings').hidden=!$('preview_enabled').checked;updatePreviewEstimate() }
const previewSeen={},previewUrls={};async function updatePreviewCards(states){for(let i=1;i<=4;i++){const state=states[String(i)],meta=$('preview-meta-'+i),img=$('preview-image-'+i),placeholder=$('preview-placeholder-'+i),enabled=$('preview_enabled').checked&&$('preview_'+i+'_enabled').checked;if(!state){meta.textContent=enabled?'Waiting for the first saved checkpoint.':'No preview generated yet.';continue}if(state.status==='loading'||state.status==='generating'||state.status==='saving'){const label={loading:'Loading Turbo',generating:'Generating',saving:'Saving image'}[state.status];meta.textContent=`${label} · step ${state.step||'—'}`;continue}if(state.status==='error'){meta.textContent='Preview failed · '+(state.error||'see Activity log');continue}if(state.status==='ready'){meta.textContent=`Step ${state.step} · seed ${state.seed} · ${Number(state.seconds||0).toFixed(1)}s${enabled?'':' · card off'}`;const key=String(state.updated_at)+'|'+state.step+'|'+state.seed;if(previewSeen[i]!==key){previewSeen[i]=key;try{const response=await fetch('/api/preview?slot='+i+'&v='+encodeURIComponent(key),{headers:{'X-Krea-Key':KEY},cache:'no-store'});if(!response.ok)throw new Error('Image fetch · HTTP '+response.status);const url=URL.createObjectURL(await response.blob()),old=previewUrls[i];img.onload=()=>{if(old)URL.revokeObjectURL(old)};img.src=url;img.hidden=false;placeholder.hidden=true;previewUrls[i]=url}catch(e){meta.textContent='Saved image could not load · '+e.message}}}}}
function setConnection(state,label){$('connection').dataset.state=state;$('connection-label').textContent=label}
async function api(path,body){const controller=new AbortController(),timer=setTimeout(()=>controller.abort(),15000),opts={method:body?'POST':'GET',headers:{'X-Krea-Key':KEY},signal:controller.signal};if(body){opts.headers['Content-Type']='application/json';opts.body=JSON.stringify(body)}try{const r=await fetch(path,opts);if(!r.ok){const e=new Error((await r.text()).slice(0,250)||('HTTP '+r.status));e.status=r.status;throw e}return await r.json()}finally{clearTimeout(timer)}}
function applyState(s){$('status').innerHTML=s.status;$('progress').innerHTML=s.progress+'<div class="bar"><i style="width:'+((s.progress.match(/(\d+)%/)||[])[1]||0)+'%"></i></div>';$('system').innerHTML=s.system;$('timing').innerHTML=s.timing;const logBox=$('logs'),wasBottom=logBox.scrollHeight-logBox.scrollTop-logBox.clientHeight<36,doc=new DOMParser().parseFromString(s.logs,'text/html');logBox.textContent=doc.body.textContent;if(wasBottom)logBox.scrollTop=logBox.scrollHeight;$('lora-summary').innerHTML=s.results;renderLoraFiles(s.loras||[]);$('dataset').innerHTML=s.dataset||'No dataset selected yet.';updatePreviewCards(s.previews||{});busy=s.busy;document.querySelectorAll('.job').forEach(b=>b.disabled=busy)}
let pollBusy=false,pollDelay=1000,pollTimer=null;async function poll(){if(pollBusy)return;pollBusy=true;try{applyState(await api('/api/state'));pollDelay=1000;setConnection('online','Studio connected · '+new Date().toLocaleTimeString([],{hour:'2-digit',minute:'2-digit',second:'2-digit'}))}catch(e){const expired=e.status===401||e.status===403;setConnection(expired?'offline':'reconnecting',expired?'Session link expired · reopen from Colab':'Reconnecting in '+Math.ceil(pollDelay/1000)+'s');pollDelay=Math.min(10000,pollDelay*2)}finally{pollBusy=false;clearTimeout(pollTimer);pollTimer=setTimeout(poll,pollDelay)}}
async function runAction(action){$('action-message').textContent='';try{await api('/api/action',{action:action,config:config()});await poll()}catch(e){$('action-message').textContent=e.message;$('action-message').classList.add('err')}}
async function usePath(){$('action-message').textContent='';try{await api('/api/dataset/path',{path:$('dataset_path').value,config:config()});await poll()}catch(e){$('action-message').textContent=e.message;$('action-message').classList.add('err')}}
async function uploadZip(){
 const f=$('zip').files[0];if(!f){$('uploadlabel').textContent='Choose a ZIP first.';return}if(!f.size){$('uploadlabel').textContent='The selected file is empty.';return}
 const total=f.size,chunkSize=8*1024*1024,id=(crypto.randomUUID?crypto.randomUUID():Date.now()+'_'+Math.random().toString(36).slice(2));
 async function sendChunk(start,end){
  for(let attempt=0;attempt<4;attempt++){
   try{return await new Promise((resolve,reject)=>{
    const x=new XMLHttpRequest();x.open('POST','/api/dataset/upload');x.timeout=300000;x.setRequestHeader('X-Krea-Key',KEY);x.setRequestHeader('X-Krea-Config',encodeURIComponent(JSON.stringify(config())));x.setRequestHeader('X-Filename',encodeURIComponent(f.name));x.setRequestHeader('X-Upload-Id',id);x.setRequestHeader('X-Upload-Offset',String(start));x.setRequestHeader('X-Upload-Total',String(total));
    x.upload.onprogress=e=>{if(e.lengthComputable){const done=start+e.loaded,p=Math.round(100*done/total);$('uploadbar').style.width=p+'%';$('uploadlabel').textContent='Uploading · '+(done/1048576).toFixed(1)+' / '+(total/1048576).toFixed(1)+' MB'}};
    x.onload=()=>{let v={};try{v=JSON.parse(x.responseText)}catch(e){}if(x.status>=300){const error=new Error(v.error||'Upload failed · HTTP '+x.status);error.retryable=x.status>=500;reject(error)}else resolve(v)};
    x.onerror=()=>reject(new Error('Upload connection failed.'));x.ontimeout=()=>reject(new Error('Upload timed out.'));x.send(f.slice(start,end))
   })}catch(error){
    if(attempt===3||error.retryable===false)throw error;
    const seconds=2**attempt;$('uploadlabel').textContent='Connection interrupted · retrying this chunk in '+seconds+'s…';
    await new Promise(resolve=>setTimeout(resolve,seconds*1000))
   }
  }
 }
 try{for(let start=0;start<total;start+=chunkSize){await sendChunk(start,Math.min(total,start+chunkSize))}$('uploadbar').style.width='100%';$('uploadlabel').textContent='ZIP received · validating images…';await poll()}catch(e){$('uploadlabel').textContent=e.message}
}
let loraSignature="";
function escapeHtml(value){return String(value).replace(/[&<>"']/g,ch=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[ch]))}
function renderLoraFiles(items){const signature=JSON.stringify(items);if(signature===loraSignature)return;loraSignature=signature;const list=$('lora-list');$('lora-download-all').disabled=!items.length;if(!items.length){list.innerHTML='<div class="lora-empty">No saved LoRA yet. Files will appear here after a checkpoint is saved.</div>';return}list.innerHTML=items.map(item=>`<div class="lora-row"><div class="lora-info"><strong>${escapeHtml(item.label)}</strong><span>${escapeHtml(item.filename)} · ${escapeHtml(item.size_label)}</span></div><button class="secondary" type="button" data-lora-file="${escapeHtml(item.filename)}">Download</button></div>`).join('')}
async function refreshLoraList(){const button=$('lora-refresh'),message=$('lora-message');button.disabled=true;message.textContent='Refreshing saved files…';try{const data=await api('/api/loras');renderLoraFiles(data.files||[]);$('lora-summary').innerHTML=data.summary||$('lora-summary').innerHTML;message.textContent=`List updated · ${(data.files||[]).length} file(s)`}catch(e){message.textContent=e.message;message.classList.add('err')}finally{button.disabled=false}}
async function startLoraDownload(payload){const message=$('lora-message');message.classList.remove('err');message.textContent=payload.kind==='all'?'Preparing ZIP…':'Preparing download…';try{const data=await api('/api/download-ticket',payload),a=document.createElement('a');a.href='/api/download?ticket='+encodeURIComponent(data.ticket);a.download=data.filename;document.body.appendChild(a);a.click();a.remove();message.textContent='Download started · '+data.filename}catch(e){message.textContent=e.message;message.classList.add('err')}}
function downloadAllLoras(){startLoraDownload({kind:'all'})}
$('lora-list').addEventListener('click',event=>{const button=event.target.closest('[data-lora-file]');if(button)startLoraDownload({kind:'file',filename:button.dataset.loraFile})});
$('lora-refresh').addEventListener('click',refreshLoraList);$('lora-download-all').addEventListener('click',downloadAllLoras);
let metricBusy=false;async function refreshMetrics(){if(metricBusy)return;metricBusy=true;try{const s=await api('/api/metrics');$('system').innerHTML=s.html}catch(e){}finally{metricBusy=false;setTimeout(refreshMetrics,2000)}}
document.addEventListener('visibilitychange',()=>{if(!document.hidden){clearTimeout(pollTimer);poll();refreshMetrics();refreshLoraList()}});
$('preview_enabled').addEventListener('change',updatePreviewControls);for(let i=1;i<=4;i++){['enabled','prompt','seed','random'].forEach(k=>$('preview_'+i+'_'+k).addEventListener('input',updatePreviewEstimate));$('preview_'+i+'_enabled').addEventListener('change',updatePreviewEstimate)}
if(KEY){updatePreviewControls();poll();refreshMetrics()}else{updatePreviewControls();setConnection('offline','Open the temporary Studio link from Colab');$('status').textContent='Open the private Studio link printed by the notebook.'}
</script></body></html>'''


def run_web_ui(studio):
    import secrets
    import urllib.parse
    api_key = secrets.token_urlsafe(24)
    job_lock = threading.Lock()
    job_running = {"value": False}
    upload_lock = threading.Lock()
    uploads = {}
    download_tickets = {}
    download_ticket_lock = threading.Lock()
    download_archive_lock = threading.Lock()

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *_args):
            pass

        def send_bytes(self, code, body, content_type="application/json"):
            if isinstance(body, str):
                body = body.encode("utf-8")
            self.send_response(code)
            self.send_header("Content-Type", content_type)
            self.send_header("Content-Length", str(len(body)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(body)

        def send_json(self, code, value):
            self.send_bytes(code, json.dumps(value, ensure_ascii=False), "application/json; charset=utf-8")

        def authorized(self):
            import hmac
            return hmac.compare_digest(self.headers.get("X-Krea-Key", ""), api_key)

        def read_json(self):
            size = int(self.headers.get("Content-Length", "0"))
            if size > 1_000_000:
                raise ValueError("Request is too large.")
            return json.loads(self.rfile.read(size) or b"{}")

        def send_lora_file(self, target):
            size = target.stat().st_size
            self.send_response(200)
            self.send_header("Content-Type", "application/octet-stream")
            self.send_header("Content-Length", str(size))
            self.send_header("Content-Disposition", f'attachment; filename="{target.name}"')
            self.send_header("Cache-Control", "no-store")
            self.send_header("X-Content-Type-Options", "nosniff")
            self.end_headers()
            try:
                with target.open("rb") as stream:
                    shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
            except (BrokenPipeError, ConnectionResetError):
                pass

        def apply_config(self, values):
            mapping = {"project":"project", "trigger":"trigger", "caption":"caption", "use_txt":"use_txt",
                "test_mode":"test_mode", "resolution":"resolution", "steps":"steps", "rank":"rank",
                "lr":"lr", "save_every":"save_every", "max_seq":"max_seq", "attention_only":"attention_only",
                "grad_ckpt":"grad_ckpt", "seed":"seed", "push":"push", "hub_id":"hub_id",
                "private":"private", "keep":"keep", "drive":"drive", "preview_enabled":"preview_enabled",
                "preview_resolution":"preview_resolution", "preview_cleanup_cache":"preview_cleanup_cache"}
            for slot in range(1, 5):
                for suffix in ("enabled", "prompt", "seed", "random"):
                    key = f"preview_{slot}_{suffix}"
                    mapping[key] = key
            for key, attr in mapping.items():
                if key in values:
                    current = getattr(studio, attr).value
                    value = values[key]
                    if isinstance(current, bool):
                        value = bool(value)
                    elif isinstance(current, int):
                        value = int(value)
                    elif isinstance(current, float):
                        value = float(value)
                    else:
                        value = str(value)
                    getattr(studio, attr).value = value

        def do_GET(self):
            route = urlparse(self.path).path
            query = parse_qs(urlparse(self.path).query)
            if route == "/":
                self.send_bytes(200, PAGE, "text/html; charset=utf-8")
            elif route == "/api/download" and "ticket" in query:
                ticket_values = query.get("ticket", [])
                if len(ticket_values) != 1:
                    self.send_json(404, {"error":"Download link is invalid or expired."})
                    return
                with download_ticket_lock:
                    entry = download_tickets.pop(ticket_values[0], None)
                if not entry or entry["expires_at"] < time.time():
                    self.send_json(404, {"error":"Download link is invalid, expired, or already used. Click Download again."})
                    return
                target = Path(entry["path"])
                try:
                    listed = studio.resolve_lora_file(entry["filename"]) if entry.get("kind") == "file" else target
                    if listed.resolve() != target.resolve() or not target.is_file() or target.stat().st_size <= 0:
                        raise FileNotFoundError
                    self.send_lora_file(target)
                except (OSError, ValueError, FileNotFoundError):
                    self.send_json(404, {"error":"This LoRA is no longer available. Refresh the list and try again."})
            elif not self.authorized():
                self.send_json(401, {"error":"Open the private session link printed by the notebook."})
            elif route == "/api/state":
                self.send_json(200, studio.state())
            elif route == "/api/loras":
                studio.refresh_results()
                self.send_json(200, {"files":studio.list_loras(), "summary":studio.results.value})
            elif route == "/api/preview":
                slot_values = query.get("slot", [])
                if len(slot_values) != 1 or slot_values[0] not in {"1", "2", "3", "4"}:
                    self.send_json(400, {"error":"Preview slot must be between 1 and 4."})
                    return
                image = studio._root() / "output" / "previews" / f"slot_{slot_values[0]}" / "latest.png"
                if not image.is_file():
                    self.send_json(404, {"error":"This preview card has no generated image yet."})
                    return
                self.send_bytes(200, image.read_bytes(), "image/png")
            elif route == "/api/metrics":
                studio._render_resources()
                self.send_json(200, {"html":studio.resources.value})
            else:
                self.send_json(404, {"error":"Not found"})

        def do_POST(self):
            if not self.authorized():
                self.send_json(401, {"error":"Session link is missing or expired."})
                return
            route = urlparse(self.path).path
            try:
                if route == "/api/download-ticket":
                    payload = self.read_json()
                    kind = payload.get("kind", "file")
                    output = studio._root() / "output"
                    if kind == "all":
                        with download_archive_lock:
                            studio.refresh_results()
                            items = studio.list_loras()
                            if not items:
                                raise ValueError("No saved LoRA files yet.")
                            target = output / f"{studio._root().name}_lora_versions.zip"
                            pending = target.with_name("." + target.name + "." + secrets.token_hex(6) + ".pending")
                            try:
                                with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_STORED) as archive:
                                    for item in items:
                                        path = studio.resolve_lora_file(item["filename"])
                                        archive.write(path, arcname=path.name)
                                os.replace(pending, target)
                            finally:
                                pending.unlink(missing_ok=True)
                        filename = target.name
                        ticket_kind = "archive"
                    elif kind == "file":
                        filename = payload.get("filename", "")
                        target = studio.resolve_lora_file(filename)
                        ticket_kind = "file"
                    else:
                        raise ValueError("Choose one saved LoRA or Download all.")
                    if not target.is_file() or target.stat().st_size <= 0:
                        raise FileNotFoundError("The selected download is missing or empty.")
                    ticket = secrets.token_urlsafe(24)
                    with download_ticket_lock:
                        now = time.time()
                        for expired in [key for key, value in download_tickets.items() if value["expires_at"] < now]:
                            download_tickets.pop(expired, None)
                        download_tickets[ticket] = {"path":str(target.resolve()), "filename":filename,
                                                    "kind":ticket_kind, "expires_at":now + 300}
                    self.send_json(200, {"ticket":ticket, "filename":filename, "expires_in":300})
                elif route == "/api/dataset/upload":
                    cfg = json.loads(urllib.parse.unquote(self.headers.get("X-Krea-Config", "%7B%7D")))
                    self.apply_config(cfg)
                    size = int(self.headers.get("Content-Length", "0"))
                    total = int(self.headers.get("X-Upload-Total", "0"))
                    offset = int(self.headers.get("X-Upload-Offset", "0"))
                    if size < 1 or size > 40 * 2**20 or total < size or total > 5 * 2**30:
                        raise ValueError("Upload chunks must be under 40 MB and ZIPs under 5 GB.")
                    filename = Path(urllib.parse.unquote(self.headers.get("X-Filename", "dataset.zip"))).name
                    if Path(filename).suffix.lower() != ".zip":
                        raise ValueError("Choose a .zip file.")
                    upload_id = self.headers.get("X-Upload-Id", "")
                    if not re.fullmatch(r"[a-zA-Z0-9_-]{8,80}", upload_id):
                        raise ValueError("Invalid upload session.")
                    with upload_lock:
                        entry = uploads.get(upload_id)
                        if offset == 0 and entry is None:
                            entry = {"path":Path("/content") / ("krea_upload_" + secrets.token_hex(8) + ".zip"),
                                     "total":total, "received":0, "chunks":{}, "processed":False}
                            uploads[upload_id] = entry
                        if entry is None or entry["total"] != total:
                            raise ValueError("Upload session expired. Start the upload again.")
                        if not hasattr(entry["path"], "parent"):
                            raise ValueError("Upload session is invalid.")
                        if offset != entry["received"] and not (offset < entry["received"] and offset in entry["chunks"]):
                            raise ValueError("Upload chunk is out of order. Start the upload again.")
                        digest = hashlib.sha256()
                        append_chunk = offset == entry["received"]
                        mode = ("r+b" if entry["path"].exists() else "wb") if append_chunk else "rb"
                        with entry["path"].open(mode) as stream:
                            stream.seek(offset)
                            remaining = size
                            while remaining:
                                chunk = self.rfile.read(min(1024 * 1024, remaining))
                                if not chunk:
                                    raise ValueError("Upload ended early.")
                                digest.update(chunk)
                                if append_chunk:
                                    stream.write(chunk)
                                remaining -= len(chunk)
                        chunk_hash = digest.hexdigest()
                        if append_chunk:
                            entry["chunks"][offset] = {"size":size, "sha256":chunk_hash}
                            entry["received"] += size
                        elif entry["chunks"].get(offset) != {"size":size, "sha256":chunk_hash}:
                            raise ValueError("Retried upload chunk did not match the accepted data.")
                        received = entry["received"]
                        uploaded = entry["path"]
                        complete = received == total
                        processed = entry.get("processed", False)
                    if not complete:
                        self.send_json(202, {"ok":True, "received":received, "total":total})
                        return
                    if not processed:
                        studio.dataset_path.value = str(uploaded)
                        studio._use_path()
                        with upload_lock:
                            uploads[upload_id]["processed"] = True
                    self.send_json(200, {"ok":True, "received":received, "total":total})
                elif route == "/api/dataset/path":
                    body = self.read_json()
                    self.apply_config(body.get("config", {}))
                    studio.dataset_path.value = body.get("path", "")
                    studio._use_path()
                    self.send_json(200, {"ok":True})
                elif route == "/api/action":
                    body = self.read_json()
                    self.apply_config(body.get("config", {}))
                    action = body.get("action", "")
                    actions = {"cache":studio._prepare_cache, "train":lambda:studio._train(False),
                               "resume":lambda:studio._train(True), "sync":studio._sync}
                    if action not in actions:
                        raise ValueError("Unknown training action.")
                    with job_lock:
                        if job_running["value"]:
                            raise ValueError("A training stage is already running.")
                        job_running["value"] = True
                    def worker():
                        try:
                            studio._start(action, actions[action])
                        finally:
                            with job_lock:
                                job_running["value"] = False
                    threading.Thread(target=worker, daemon=True, name="krea2-stage").start()
                    self.send_json(202, {"ok":True})
                else:
                    self.send_json(404, {"error":"Not found"})
            except Exception as exc:
                self.send_json(400, {"error":str(exc)})

    server = ThreadingHTTPServer(("127.0.0.1", 7860), Handler)
    server.daemon_threads = True
    threading.Thread(target=server.serve_forever, daemon=True, name="krea2-web").start()
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return "http://127.0.0.1:7860/#" + api_key, server, None

    binary = shutil.which("cloudflared")
    if not binary:
        binary = "/content/cloudflared"
        import urllib.request
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
        os.chmod(binary, 0o755)
    tunnel = subprocess.Popen([binary, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    import selectors
    selector = selectors.DefaultSelector()
    selector.register(tunnel.stdout, selectors.EVENT_READ)
    pending, public_url, deadline = "", None, time.monotonic() + 90
    while time.monotonic() < deadline and tunnel.poll() is None and not public_url:
        if selector.select(timeout=1):
            chunk = os.read(tunnel.stdout.fileno(), 4096).decode("utf-8", errors="replace")
            pending += chunk
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", pending, re.I)
            if match:
                public_url = match.group(0)
    selector.close()
    if not public_url:
        tunnel.terminate()
        server.shutdown()
        raise RuntimeError("Cloudflare quick tunnel did not return a URL within 90 seconds.")
    return public_url + "/#" + api_key, server, tunnel


try:
    KREA_STUDIO.close()
except NameError:
    pass
KREA_STUDIO = KreaStudio()
try:
    KREA_SERVER.shutdown()
    KREA_SERVER.server_close()
except Exception:
    pass
try:
    KREA_TUNNEL.terminate()
except Exception:
    pass
KREA_URL, KREA_SERVER, KREA_TUNNEL = run_web_ui(KREA_STUDIO)
display(HTML(f"<div style='font:16px Arial;padding:20px;border:1px solid #ddd;border-radius:10px'>"
    f"<b>Krea 2 LoRA Studio is ready.</b><br><br>Open the control panel in a new tab: "
    f"<a href='{html.escape(KREA_URL, quote=True)}' target='_blank' rel='noopener'>Open Krea 2 Studio</a>"
    "<br><small>The temporary session link stays active while this Colab runtime is running.</small></div>"))
